# PHOENIX-2014-T CSLR — Pipeline Migliorata
## DenseNet121 + BiLSTM + CTC con tutte le ottimizzazioni dai paper

### Miglioramenti rispetto alla versione originale:
1. **Two-phase finetuning** (Deep Sign §6.2) — backbone congelato in fase 1, LR differenziale in fase 2
2. **Prior scaling β** (Deep Sign Eq.13) — sottrae log-prior pesato prima del beam search
3. **Model ensemble** (Deep Sign §6.5) — media log-prob degli ultimi N checkpoint (+6% WER relativo)
4. **Più layer LSTM** (approssima più stati HMM, Deep Sign §6.4)
5. **Gloss vocabulary reduction** — analisi e merging di classi rare/simili
6. **Fix OOM** — batch size ridotto + gradient checkpointing sul DenseNet
7. **Fix bug** — duplicato `generate_tssi_optimized`, `try` annidato irraggiungibile
8. **Checkpoint frequenti** negli ultimi 10 epoch per ensemble

In [1]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter, defaultdict
from difflib import SequenceMatcher
from scipy.ndimage import gaussian_filter1d
from scipy.interpolate import interp1d

# ===============================================
# CONFIGURAZIONE PERCORSI KAGGLE - DEFINITIVA
# ===============================================

# La radice del tuo dataset importato
BASE_PATH = "/home/ebufi/phoenix"

# 1. Cartella CSV (Confermata)
ANNOTATIONS_DIR = os.path.join(BASE_PATH, 'annotations','annotations', 'manual')

# 2. Cartella Keypoint (Confermata dal tuo path!)
# Puntiamo alla cartella padre che contiene 'train', 'dev' e 'test'
POSE_DIR = os.path.join(BASE_PATH, 'pose', 'pose') 

# 3. Cartelle di Output in /kaggle/working/ (scrivibile)
RESULTS_DIR     = '/home/ebufi/phoenix/results'
TSSI_OUTPUT_DIR = '/home/ebufi/phoenix/tssi_cache' 

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(TSSI_OUTPUT_DIR, exist_ok=True)

print("✓ Percorsi Kaggle configurati perfettamente!")
print(f"Annotazioni: {ANNOTATIONS_DIR}")
print(f"Keypoint: {POSE_DIR}")

✓ Percorsi Kaggle configurati perfettamente!
Annotazioni: /home/ebufi/phoenix/annotations/annotations/manual
Keypoint: /home/ebufi/phoenix/pose/pose


---
## Step 1 — Import e Configurazione

In [2]:
import os
import gc
import json
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from torch.utils.checkpoint import checkpoint as grad_checkpoint
from torchvision import models
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from collections import Counter, defaultdict
from scipy.ndimage import gaussian_filter1d
import cv2
import warnings
warnings.filterwarnings('ignore')

print('✓ Import completati')

✓ Import completati


In [3]:
# ============================================================================
# 🔧 FUNZIONI OTTIMIZZATE DA optimized_code.py
# ============================================================================

# FIX 1: Memory-Efficient Data Augmentation
def augment_tssi_fixed(tssi: np.ndarray, augment: bool = True) -> np.ndarray:
    if not augment:
        return tssi
    C, J, T = tssi.shape

    # 1. Jitter gaussiano
    if np.random.rand() < 0.8:
        noise = gaussian_filter1d(
            np.random.randn(2, J, T).astype(np.float32), sigma=3, axis=2)
        tssi[:2] = np.clip(tssi[:2] + noise * 0.008, 0, 1)
        del noise

    # 2. Temporal warping — vettorizzato (no loop su joint)
    if np.random.rand() < 0.7:
        speeds  = np.random.uniform(0.7, 1.3, T).astype(np.float32)
        indices = np.cumsum(speeds)
        indices = (indices / indices[-1] * (T - 1)).astype(np.float32)
        src     = np.arange(T, dtype=np.float32)
        warped  = np.stack([
            np.clip(np.array([np.interp(indices, src, tssi[c, j]) for j in range(J)]), 0, 1)
            for c in range(C)
        ])
        tssi = warped
        del speeds, indices, warped

    # 3. Spatial noise
    if np.random.rand() < 0.5:
        spatial_noise = np.random.randn(2, J, T).astype(np.float32) * 0.005
        tssi[:2] = np.clip(tssi[:2] + spatial_noise, 0, 1)
        del spatial_noise

    # 4. Time masking (SpecAugment) — NEW
    if np.random.rand() < 0.7:
        for _ in range(2):
            t_len = np.random.randint(1, max(2, int(T * 0.15)))
            t_start = np.random.randint(0, max(1, T - t_len))
            tssi[:, :, t_start:t_start + t_len] = 0.0

    # 5. Joint masking — NEW
    if np.random.rand() < 0.5:
        mask = np.random.rand(J) < 0.15
        tssi[:, mask, :] = 0.0

    return tssi

# FIX 2: Correct DFS Order Generation
def get_dfs_order_phoenix_correct() -> np.ndarray:
    """Constructs correct DFS order for MediaPipe skeleton (48 keypoints)."""
    adj = defaultdict(list)
    edges_body = [(0,1),(1,2),(2,3),(1,4),(4,5)]
    edges_wrist_to_hand = [(3,6),(5,27)]
    finger_edges = [
        (0,1),(1,2),(2,3),(3,4),
        (0,5),(5,6),(6,7),(7,8),
        (0,9),(9,10),(10,11),(11,12),
        (0,13),(13,14),(14,15),(15,16),
        (0,17),(17,18),(18,19),(19,20),
    ]
    edges_lh = [(s+6,  e+6)  for s,e in finger_edges]
    edges_rh = [(s+27, e+27) for s,e in finger_edges]
    
    for u,v in edges_body + edges_wrist_to_hand + edges_lh + edges_rh:
        if u < 48 and v < 48:
            adj[u].append(v)
            adj[v].append(u)
    
    dfs_path, visited = [], set()
    def dfs(node):
        dfs_path.append(node)
        visited.add(node)
        for n in sorted([x for x in adj[node] if x not in visited]):
            dfs(n)
            if len(dfs_path) < 135:
                dfs_path.append(node)
    
    dfs(0)
    while len(dfs_path) < 135:
        dfs_path.extend(dfs_path[-10:])
    
    return np.array(dfs_path[:135], dtype=np.int32)


# FIX 3: Numerically Stable CTC Loss with Entropy Regularization
class CTCLossWithEntropy(nn.Module):
    def __init__(self, blank=0, entropy_weight=0.05, class_weights=None):
        super().__init__()
        self.blank          = blank
        self.entropy_weight = entropy_weight
        self.class_weights  = class_weights   # ← aggiunto
        self.ctc = nn.CTCLoss(blank=blank, reduction='mean', zero_infinity=True)

    def forward(self, log_probs_TBC, targets, input_lengths, target_lengths):
        T, B, C = log_probs_TBC.shape

        # Applica class weights come prior scaling pre-softmax
        if self.class_weights is not None:
            log_w = torch.log(self.class_weights.clamp(min=1e-8))
            log_probs_TBC = log_probs_TBC + log_w.unsqueeze(0).unsqueeze(0)
            # Ri-normalizza per mantenere distribuzione valida
            log_probs_TBC = log_probs_TBC - log_probs_TBC.logsumexp(dim=-1, keepdim=True)

        ctc_loss = self.ctc(log_probs_TBC, targets, input_lengths, target_lengths)

        if self.entropy_weight > 0:
            probs      = torch.exp(log_probs_TBC).clamp(min=1e-10)
            entropy    = -(probs * torch.log(probs)).sum(dim=-1).mean()
            max_entropy = np.log(C)
            entropy_reg = entropy / max(max_entropy, 1e-8)
            return ctc_loss - self.entropy_weight * entropy_reg   # ← segno - (Fix 1)

        return ctc_loss

# FIX 4: Optimized Beam Search CTC Decoding with Prior Scaling
def beam_search_ctc_optimized(
    log_probs_TBC: torch.Tensor,
    beam_width: int = 10,
    beta: float = 0.3,
    log_prior: torch.Tensor = None
):
    """Beam search CTC decoding with prior scaling (Deep Sign Eq. 13)."""
    from scipy.special import logsumexp
    
    T, B, C = log_probs_TBC.shape
    results = []
    
    for b in range(B):
        lp = log_probs_TBC[:, b, :].cpu().numpy()
        
        # Prior scaling
        if beta > 0.0 and log_prior is not None:
            lp_prior = log_prior.cpu().numpy()
            lp = lp - beta * lp_prior.reshape(1, -1)
            lp = lp - logsumexp(lp, axis=1, keepdims=True)
        blank_penalty = 0.3   # cerca in [0.1, 0.5] sul dev
        lp[:, 0] -= blank_penalty
        # Beam search
        beams = {((), None): 0.0}
        
        for t in range(T):
            new_beams = {}
            for (prefix, last_char), beam_lp in beams.items():
                for c in range(C):
                    new_lp = beam_lp + lp[t, c]
                    
                    if c == 0:
                        key = (prefix, last_char)
                    elif c == last_char:
                        continue
                    else:
                        key = (prefix + (c,), c)
                    
                    if key not in new_beams or new_beams[key] < new_lp:
                        new_beams[key] = new_lp
            
            sorted_beams = sorted(new_beams.items(), key=lambda x: -x[1])
            beams = dict(sorted_beams[:beam_width])
        
        best_key = max(beams.keys(), 
               key=lambda k: beams[k] / max(len(k[0]), 1) ** 0.7)
        best_seq = list(best_key[0])
        results.append(best_seq)
    
    return results


# FIX 5: Learning Rate Scheduler with Warmup + Cosine Annealing
def create_scheduler_warmup_cosine(
    optimizer,
    total_steps: int,
    warmup_steps: int,
    min_lr_ratio: float = 0.0
) -> torch.optim.lr_scheduler.LambdaLR:
    """Creates scheduler with linear warmup + cosine annealing."""
    
    def lr_lambda(current_step: int) -> float:
        if current_step < warmup_steps:
            return float(current_step) / float(max(1, warmup_steps))
        else:
            progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
            cosine_decay = 0.5 * (1.0 + np.cos(np.pi * progress))
            return max(min_lr_ratio, cosine_decay)
    
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# FIX 6: Checkpoint Manager (Keep Only Top-K)
class CheckpointManager:
    """Saves only top-K checkpoints by metric, deletes rest."""
    
    def __init__(self, save_dir: str, keep_top_k: int = 3, metric: str = 'wer'):
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.keep_top_k = keep_top_k
        self.metric = metric
        self.checkpoints = []
    
    def save(self, model: nn.Module, epoch: int, metrics: dict) -> Path:
        metric_value = metrics[self.metric]
        path = self.save_dir / f"epoch_{epoch:03d}_{self.metric}_{metric_value:.4f}.pth"
        torch.save(model.state_dict(), path)
        
        self.checkpoints.append((metric_value, path, epoch))
        self.checkpoints.sort(key=lambda x: x[0])
        
        if len(self.checkpoints) > self.keep_top_k:
            _, worst_path, worst_epoch = self.checkpoints.pop()
            if worst_path.exists():
                worst_path.unlink()
        
        return path
    
    def get_best_path(self) -> Path:
        if not self.checkpoints:
            return None
        return self.checkpoints[0][1]


# FIX 7: Run Epoch with Correct Gradient Accumulation
def run_epoch_with_correct_accumulation(
    model: nn.Module,
    dataloader,
    criterion: nn.Module,
    optimizer,
    scaler: GradScaler,
    scheduler,
    training: bool,
    device,
    config: dict,
    log_prior: torch.Tensor
):
    """Training/validation loop with CORRECT gradient accumulation."""
    
    model.train() if training else model.eval()
    total_loss = 0.0
    all_refs, all_hyps = [], []
    
    accum_steps = config['gradient_accumulation_steps']
    grad_clip = config['grad_clip']
    use_amp = config['use_amp'] and device.type == 'cuda'
    
    if training:
        optimizer.zero_grad(set_to_none=True)
    
    for batch_idx, (tssies, targets, input_lengths, target_lengths) in enumerate(
            tqdm(dataloader, leave=False, desc='Train' if training else 'Val')):
        
        tssies = tssies.to(device, non_blocking=True)
        
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            log_probs = model(tssies)
            lp_float = log_probs.permute(1, 0, 2).float()
            
            T_out = lp_float.shape[0]
            T_in = tssies.shape[3]
            scale = T_out / T_in
            
            ctc_input_lengths = (input_lengths.float() * scale).long().clamp(1, T_out).cpu()
            ctc_target_lengths = target_lengths.to(dtype=torch.long, device='cpu')
            ctc_targets = targets.to(dtype=torch.long, device='cpu')
            
            loss = criterion(lp_float, ctc_targets, ctc_input_lengths, ctc_target_lengths)
        
        total_loss += loss.item()
        
        if training:
            scaled_loss = loss / accum_steps
            scaler.scale(scaled_loss).backward()
            
            if (batch_idx + 1) % accum_steps == 0 or (batch_idx + 1) == len(dataloader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                
                if scheduler is not None:
                    scheduler.step()
    
    avg_loss = total_loss / len(dataloader)
    return avg_loss, 0.0


print('✓ Funzioni ottimizzate importate')

✓ Funzioni ottimizzate importate


In [ ]:
# ============================================================================
# ============================================================================
# CONFIGURAZIONE E PERCORSI (UNIFICATO PER KAGGLE)

# ============================================================================
CONFIG = {
    # Training generale
    'num_epochs':               150,
    'early_stopping_patience':  20,
    'checkpoint_every':         1,
    'keep_last_n_checkpoints':  10,
   
   
    
    # Batch e ottimizzazione
    'batch_size':                    16,
    'gradient_accumulation_steps':   1,
    'use_amp':                       True,
    'hidden_dim': 256,
    'tcn_blocks': 3,
    'num_layers': 3,
    'dropout': 0.317638942624273,
    'weight_decay': 0.0002465667957406484,
    'grad_clip': 3.3943208852931304,
    'prior_beta': 0.32404536014294083,
    'ctc_smoothing': 0.10231111042825411,
    'phase1_lr': 0.0001619145621568752,
    'phase2_lr_backbone': 0.00021884694140182422,
    'phase2_lr_head': 0.0005038675815696706,
    'overfitting_lambda': 1.1575402465259297,
    
    
    
    'num_joints':    48,
    'num_channels':  5,
    

    # Two-phase finetuning
    'phase1_epochs': 15,
    

    # Prior scaling
    
    'drop_path_rate': 0.1,
    'attn_heads':     4,
    # Ensemble
    'ensemble_n':    3,

    # CTC
   
    'beam_width':    25,

    # Input
    'frame_h':       240,

    # Gloss merging
    'use_gloss_merge': True,
    'merge_map_path':  '/kaggle/working/gloss_merge_map.json',

    # Debug training
    'debug_training':        False,
    'debug_every_n_batches':  28,
    'debug_preview_batches':  2,
    'debug_preview_samples':  3,
    'debug_topk_classes':     8,
    'debug_save_reports':     True,
}
  
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'




Device: cuda
GPU: NVIDIA GeForce RTX 3080


In [ ]:
# ============================================================
# CUDA cleanup + Optuna search (Ultra-Sicuro & Pruning Attivo)
# ============================================================
import gc
import torch
import numpy as np
from torch.utils.data import Subset, DataLoader
from torch.cuda.amp import GradScaler

import gc
import torch
import numpy as np
from torch.utils.data import Subset, DataLoader
from torch.cuda.amp import GradScaler

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()

try:
    import optuna
except Exception as exc:
    raise RuntimeError("Installa optuna con: !pip install optuna") from exc


def _make_subset(dataset, frac=0.5, seed=42):
    n = len(dataset)
    k = max(1, int(n * frac))
    idx = np.random.default_rng(seed).choice(n, size=k, replace=False)
    return Subset(dataset, idx)


def build_model_from_cfg(cfg, num_classes):
    return PoseNetworkCTC(
        num_classes=num_classes,
        num_joints=cfg["num_joints"],
        num_channels=cfg.get("num_channels", 3),
        hidden_dim=cfg["hidden_dim"],
        tcn_blocks=cfg["tcn_blocks"],
        lstm_layers=cfg["num_layers"],
        dropout=cfg["dropout"],
    ).to(DEVICE)


def optuna_objective(trial):
    """
    Obiettivo composito anti-overfitting:
        score = dev_wer + λ * max(0, gap - soglia)
    Penalizza esplicitamente modelli che memorizzano il training set.
    Testa entrambe le fasi del two-phase finetuning.
    """
    cleanup_cuda()
    cfg = dict(CONFIG)

    # ── Spazio di ricerca ────────────────────────────────────────────────
    cfg["hidden_dim"]        = trial.suggest_categorical("hidden_dim", [128, 192, 256])
    cfg["tcn_blocks"]        = trial.suggest_int("tcn_blocks", 3, 5)
    cfg["num_layers"]        = trial.suggest_int("num_layers", 1, 3)
    cfg["dropout"]           = trial.suggest_float("dropout", 0.25, 0.55)
    cfg["weight_decay"]      = trial.suggest_float("weight_decay", 1e-4, 5e-3, log=True)
    cfg["grad_clip"]         = trial.suggest_float("grad_clip", 1.0, 5.0)
    cfg["prior_beta"]        = trial.suggest_float("prior_beta", 0.1, 0.7)
    cfg["ctc_smoothing"]     = trial.suggest_float("ctc_smoothing", 0.03, 0.15)

    # LR separati per le due fasi
    cfg["phase1_lr"]              = trial.suggest_float("phase1_lr", 1e-4, 8e-4, log=True)
    cfg["phase2_lr_backbone"] = trial.suggest_float("phase2_lr_backbone", 1e-4, 1e-3, log=True)
    cfg["phase2_lr_head"]     = trial.suggest_float("phase2_lr_head",     1e-4, 1e-3, log=True)

    # Penalità overfitting: quanto pesare il gap train/dev sull'obiettivo
    overfitting_lambda = trial.suggest_float("overfitting_lambda", 0.3, 1.5)
    gap_tolerance      = 0.08   # tolleriamo fino a 8pp di gap train/dev — oltre si penalizza

    cfg["batch_size"] = min(cfg["batch_size"], 8)
    cfg["gradient_accumulation_steps"] = 1

    # Epoch budget: 8 fase1 + 12 fase2 = 20 totali (veloci su subset)
    PHASE1_EP = 15
    PHASE2_EP = 25

    try:
        model_trial     = build_model_from_cfg(cfg, num_classes)
        criterion_trial = CTCLossWithEntropy(blank=0, entropy_weight=cfg["ctc_smoothing"])
        scaler_trial    = GradScaler(enabled=cfg["use_amp"] and DEVICE.type == "cuda")

        train_subset = _make_subset(train_ds, frac=0.5, seed=trial.number + 7)
        dev_subset   = _make_subset(dev_ds,   frac=0.5, seed=trial.number + 11)

        train_dl_sub = DataLoader(
            train_subset, batch_size=cfg["batch_size"], shuffle=True,
            num_workers=NUM_WORKERS, pin_memory=True,
            collate_fn=collate_fn_ctc, drop_last=True)
        dev_dl_sub = DataLoader(
            dev_subset, batch_size=cfg["batch_size"], shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=True,
            collate_fn=collate_fn_ctc)

        best_score   = float("inf")
        global_epoch = 0

        # ── FASE 1: backbone congelato ────────────────────────────────────
        freeze_backbone(model_trial)
        opt_p1  = make_optimizer_phase1(model_trial, lr=cfg["phase1_lr"],
                                        weight_decay=cfg["weight_decay"])
        sch_p1  = make_scheduler(opt_p1, PHASE1_EP, len(train_dl_sub))

        for ep in range(PHASE1_EP):
            global_epoch += 1

            (train_loss, train_wer, *_) = run_epoch(
                model_trial, train_dl_sub, criterion_trial,
                opt_p1, scaler_trial, sch_p1,
                training=True, device=DEVICE, config=cfg,
                log_prior=LOG_PRIOR, i2g=i2g)

            (_, dev_wer, *_) = run_epoch(
                model_trial, dev_dl_sub, criterion_trial,
                opt_p1, scaler_trial, sch_p1,
                training=False, device=DEVICE, config=cfg,
                log_prior=LOG_PRIOR, i2g=i2g)

            gap   = max(0.0, train_wer - dev_wer)  # gap positivo = train > dev (ok)
            # gap negativo significativo = dev > train = overfitting
            overfit_gap = max(0.0, dev_wer - train_wer - gap_tolerance)
            score = dev_wer + overfitting_lambda * overfit_gap

            best_score = min(best_score, score)
            trial.report(score, global_epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

        # ── FASE 2: tutto sbloccato ───────────────────────────────────────
        unfreeze_backbone(model_trial)
        opt_p2 = make_optimizer_phase2(
            model_trial,
            lr_backbone=cfg["phase2_lr_backbone"],
            lr_head=cfg["phase2_lr_head"],
            weight_decay=cfg["weight_decay"])
        sch_p2 = make_scheduler(opt_p2, PHASE2_EP, len(train_dl_sub))

        for ep in range(PHASE2_EP):
            global_epoch += 1

            (train_loss, train_wer, *_) = run_epoch(
                model_trial, train_dl_sub, criterion_trial,
                opt_p2, scaler_trial, sch_p2,
                training=True, device=DEVICE, config=cfg,
                log_prior=LOG_PRIOR, i2g=i2g)

            (_, dev_wer, *_) = run_epoch(
                model_trial, dev_dl_sub, criterion_trial,
                opt_p2, scaler_trial, sch_p2,
                training=False, device=DEVICE, config=cfg,
                log_prior=LOG_PRIOR, i2g=i2g)

            overfit_gap = max(0.0, dev_wer - train_wer - gap_tolerance)
            score       = dev_wer + overfitting_lambda * overfit_gap

            best_score = min(best_score, score)
            trial.report(score, global_epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

        return best_score

    except torch.OutOfMemoryError:
        print(f"[Trial {trial.number}] OOM — scartato.")
        cleanup_cuda()
        return float("inf")

    except optuna.exceptions.TrialPruned:
        raise

    except Exception as e:
        print(f"[Trial {trial.number}] Errore: {e}")
        cleanup_cuda()
        return float("inf")

    finally:
        cleanup_cuda()


# ── Avvio ricerca ─────────────────────────────────────────────────────────
study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=4)
)

print("Inizio ricerca (30 trial, two-phase, obiettivo anti-overfitting)...")
study.optimize(optuna_objective, n_trials=30, show_progress_bar=True)

best = study.best_trial
print("\n" + "=" * 55)
print(f"Trial migliore: #{best.number}  |  Score: {best.value:.4f}")
print("Parametri da copiare nel CONFIG:")
for k, v in best.params.items():
    print(f"    '{k}': {v},")

# Mostra anche i trial NON potati con gap train/dev basso (i più stabili)
print("\nTop 5 trial per score (incluso gap):")
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
for t in sorted(completed, key=lambda x: x.value)[:5]:
    print(f"  Trial #{t.number:3d}  score={t.value:.4f}  "
          f"dropout={t.params.get('dropout', '?'):.2f}  "
          f"wd={t.params.get('weight_decay', '?'):.1e}  "
          f"lambda={t.params.get('overfitting_lambda', '?'):.2f}")
print("=" * 55)

# ============================================================
# AVVIO RICERCA
# ============================================================

#

[I 2026-05-26 10:58:01,459] A new study created in memory with name: no-name-60360880-fdc3-46de-b890-ff004c11ff8a


Inizio ricerca (30 trial, two-phase, obiettivo anti-overfitting)...


  0%|          | 0/30 [00:00<?, ?it/s]

[Trial 0] Errore: name 'num_classes' is not defined


Best trial: 0. Best value: inf:   3%|▎         | 1/30 [00:00<00:07,  3.81it/s]

...

[I 2026-05-26 10:58:08,247] Trial 27 finished with value: inf and parameters: {'hidden_dim': 128, 'tcn_blocks': 4, 'num_layers': 3, 'dropout': 0.4465863576189025, 'weight_decay': 0.0005184547681038936, 'grad_clip': 2.4038310790700854, 'prior_beta': 0.4439666175129026, 'ctc_smoothing': 0.10740501711430858, 'phase1_lr': 0.00029296669133493144, 'phase2_lr_backbone': 0.00018626494281905165, 'phase2_lr_head': 0.0007447377797175966, 'overfitting_lambda': 1.4309868524432274}. Best is trial 0 with value: inf.
[Trial 28] Errore: name 'num_classes' is not defined


Best trial: 0. Best value: inf:  97%|█████████▋| 29/30 [00:07<00:00,  4.11it/s]

[I 2026-05-26 10:58:08,481] Trial 28 finished with value: inf and parameters: {'hidden_dim': 192, 'tcn_blocks': 4, 'num_layers': 1, 'dropout': 0.32436711582348304, 'weight_decay': 0.0008765991482782162, 'grad_clip': 1.4671223012094472, 'prior_beta': 0.5093608311090987, 'ctc_smoothing': 0.1295459266255288, 'phase1_lr': 0.0004994569388015102, 'phase2_lr_backbone': 0.00013884145664814492, 'phase2_lr_head': 0.0002523639650494013, 'overfitting_lambda': 0.8577818113008466}. Best is trial 0 with value: inf.
[Trial 29] Errore: name 'num_classes' is not defined


Best trial: 0. Best value: inf: 100%|██████████| 30/30 [00:07<00:00,  4.13it/s]

[I 2026-05-26 10:58:08,719] Trial 29 finished with value: inf and parameters: {'hidden_dim': 256, 'tcn_blocks': 3, 'num_layers': 2, 'dropout': 0.5287259326628901, 'weight_decay': 0.0003158091233374529, 'grad_clip': 4.488015419246765, 'prior_beta': 0.583140939025273, 'ctc_smoothing': 0.14074347997053896, 'phase1_lr': 0.00019944854497129036, 'phase2_lr_backbone': 0.0004846338249055659, 'phase2_lr_head': 0.0005252972131968601, 'overfitting_lambda': 0.7783662547642536}. Best is trial 0 with value: inf.

Trial migliore: #0  |  Score: inf
Parametri da copiare nel CONFIG:
    'hidden_dim': 192,
    'tcn_blocks': 4,
    'num_layers': 1,
    'dropout': 0.47733705752549127,
    'weight_decay': 0.0006781617357729642,
    'grad_clip': 2.933003714475948,
    'prior_beta': 0.23881941516878044,
    'ctc_smoothing': 0.12988124061299589,
    'phase1_lr': 0.00011165818740118469,
    'phase2_lr_backbone': 0.00030891100074030456,
    'phase2_lr_head': 0.0001635340230912947,
    'overfitting_lambda': 1.426

## Step 2 — Caricamento dati e vocabolario

In [6]:
def load_split(split):
    return pd.read_csv(
        os.path.join(
            '/home/ebufi/phoenix/annotations/annotations/manual',
            f'PHOENIX-2014-T.{split}.corpus.csv'
        ),
        sep='|'
    )


train_df = load_split('train')
dev_df   = load_split('dev')
test_df  = load_split('test')

print(f'Train: {len(train_df)} | Dev: {len(dev_df)} | Test: {len(test_df)}')


def normalize_gloss_token(token):
    token = str(token).strip().upper()
    token = token.replace('–', '-').replace('—', '-').replace('_', '-')
    token = re.sub(r'^[^0-9A-ZÀ-ÖØ-öø-ÿÄÖÜß]+|[^0-9A-ZÀ-ÖØ-öø-ÿÄÖÜß]+$', '', token)
    token = re.sub(r'\s+', '', token)
    return token


raw_gloss_counter = Counter()
raw_glosses = set()
for df in [train_df, dev_df, test_df]:
    for gloss_seq in df['orth'].fillna(''):
        tokens = [normalize_gloss_token(tok) for tok in str(gloss_seq).split()]
        tokens = [tok for tok in tokens if tok]
        raw_gloss_counter.update(tokens)
        raw_glosses.update(tokens)

print(f'Gloss grezzi unici: {len(raw_glosses)}')
print(f'Token con trattino: {sum(1 for g in raw_glosses if "-" in g)}')
print(f'Token con cifre:    {sum(1 for g in raw_glosses if any(ch.isdigit() for ch in g))}')


merge_map = {}
if CONFIG['use_gloss_merge'] and os.path.exists(CONFIG['merge_map_path']):
    with open(CONFIG['merge_map_path'], 'r', encoding='utf-8') as f:
        merge_map = json.load(f)
    print(f'✓ Merge map caricata: {len(merge_map)} gloss → {len(set(merge_map.values()))} classi')
else:
    print('⚠ Nessuna merge map manuale trovata — userò solo normalizzazione sicura')


safe_merge_map = {}
for token in sorted(raw_glosses):
    compact = token.replace('-', '')
    if '-' in token and compact in raw_glosses and compact != token:
        safe_merge_map[token] = compact

if CONFIG['use_gloss_merge']:
    if merge_map:
        safe_only = {k: v for k, v in safe_merge_map.items() if k not in merge_map}
        if safe_only:
            merge_map.update(safe_only)
            print(f'✓ Merge map estesa con {len(safe_only)} varianti ortografiche sicure')
    else:
        merge_map = dict(safe_merge_map)
        print(f'✓ Merge map auto-generata in modo conservativo: {len(merge_map)} gloss → {len(set(merge_map.values()))} classi')

if CONFIG['debug_save_reports'] and CONFIG['use_gloss_merge'] and merge_map and not os.path.exists(CONFIG['merge_map_path']):
    with open(CONFIG['merge_map_path'], 'w', encoding='utf-8') as f:
        json.dump(merge_map, f, indent=2, ensure_ascii=False)
    print(f'✓ Merge map salvata in: {CONFIG["merge_map_path"]}')


def apply_merge(gloss_seq_str):
    merged_tokens = []
    for token in str(gloss_seq_str).strip().split():
        normalized = normalize_gloss_token(token)
        if not normalized:
            continue
        merged_tokens.append(merge_map.get(normalized, normalized))
    return merged_tokens


merged_gloss_counter = Counter()
merged_glosses = set()
for df in [train_df, dev_df, test_df]:
    for gloss_seq in df['orth'].fillna(''):
        tokens = apply_merge(gloss_seq)
        merged_gloss_counter.update(tokens)
        merged_glosses.update(tokens)

raw_gloss_count = len(raw_glosses)
merged_gloss_count = len(merged_glosses)
merged_gloss_reduction = raw_gloss_count - merged_gloss_count

print('\nGloss summary:')
print(f'  Raw gloss unici:        {raw_gloss_count}')
print(f'  Gloss unici dopo merge: {merged_gloss_count}')
print(f'  Riduzione classi:       {merged_gloss_reduction}')
print(f'  Merge sicuri applicati: {len(safe_merge_map)}')

# ── Rimozione classi rare ─────────────────────────────────────────────────
MIN_FREQ = 5
rare_glosses_set = {
    g for g, cnt in merged_gloss_counter.items()
    if cnt < MIN_FREQ and g != '<blank>'
}
print(f'Classi rare (count < {MIN_FREQ}): {len(rare_glosses_set)} → mappate a <unk>')

vocab = ['<blank>', '<unk>'] + sorted(
    g for g in merged_glosses
    if g not in rare_glosses_set and g != '<blank>'
)
g2i = {g: i for i, g in enumerate(vocab)}
i2g = {i: g for g, i in g2i.items()}

# Le glosse rare puntano all'indice di <unk>
for g in rare_glosses_set:
    g2i[g] = g2i['<unk>']

num_classes = len(vocab)
print(f'Vocabolario: {num_classes} classi ({len(rare_glosses_set)} rare → <unk>)')
g2i = {g: i for i, g in enumerate(vocab)}
i2g = {i: g for g, i in g2i.items()}
num_classes = len(vocab)

print(f"\n✓ Vocabolario consolidato:")
print(f"  Classe 0: {vocab[0]}")
print(f"  Classi 1-{len(vocab)-1}: Gloss veri")
print(f"  Totale classi: {num_classes}")

# Verifica
print(f"\nVerifica:")
print(f"  g2i['<blank>'] = {g2i.get('<blank>', 'MANCA!')}")
print(f"  i2g[0] = {i2g.get(0, 'MANCA!')}")

token_counts = Counter()
for seq in train_df['orth'].fillna(''):
    for g in apply_merge(seq):
        if g in g2i:
            token_counts[g2i[g]] += 1

prior = torch.zeros(num_classes)
for idx, cnt in token_counts.items():
    prior[idx] = cnt
prior[0] = prior[1:].sum() * 0.01  # prior blank basso ma > 0
prior = prior / prior.sum()
LOG_PRIOR = torch.log(prior.clamp(min=1e-8))

print(f'✓ Log-prior calcolato (β={CONFIG["prior_beta"]})')
class DropPath(nn.Module):
    """
    Stochastic Depth (DropPath): azzera l'intero ramo conv
    per un sottoinsieme casuale di sample nel batch.
    Equivale a "saltare" il blocco residuale durante il training,
    riducendo il co-adattamento tra blocchi TCN consecutivi.
    A inference time (eval) è un no-op esatto.
    """
    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self.training or self.drop_prob == 0.0:
            return x
        keep_prob = 1.0 - self.drop_prob
        # shape: (B, 1, 1) → si broadcast su (B, C, T)
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask = torch.rand(shape, dtype=x.dtype, device=x.device) < keep_prob
        return x * mask.float() / keep_prob
class TemporalAttention(nn.Module):
    """
    Multi-head self-attention temporale con residual + LayerNorm.
    Si interpone tra TCN e BiLSTM per catturare dipendenze
    a lungo raggio che le convoluzioni dilatate non coprono
    (es. relazione tra inizio e fine di un segno lungo).
    """
    def __init__(self, dim: int, heads: int = 4, dropout: float = 0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=heads,
            dropout=dropout,
            batch_first=True   # accetta (B, T, D)
        )
        self.norm    = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, D)
        attn_out, _ = self.attn(x, x, x)
        return self.norm(x + self.dropout(attn_out))
# Nel modello principale, inserire tra TCN e BiLSTM:
# self.temporal_attn = TemporalAttention(hidden_dim, heads=4)
# Nel forward: features = self.temporal_attn(features)  # prima del BiLSTM

Train: 7096 | Dev: 519 | Test: 642
Gloss grezzi unici: 1115
Token con trattino: 159
Token con cifre:    2
⚠ Nessuna merge map manuale trovata — userò solo normalizzazione sicura
✓ Merge map auto-generata in modo conservativo: 0 gloss → 0 classi

Gloss summary:
  Raw gloss unici:        1115
  Gloss unici dopo merge: 1115
  Riduzione classi:       0
  Merge sicuri applicati: 0
Classi rare (count < 5): 621 → mappate a <unk>
Vocabolario: 496 classi (621 rare → <unk>)

✓ Vocabolario consolidato:
  Classe 0: <blank>
  Classi 1-495: Gloss veri
  Totale classi: 496

Verifica:
  g2i['<blank>'] = 0
  i2g[0] = <blank>
✓ Log-prior calcolato (β=0.32404536014294083)


In [7]:
        # import json

        # # 1. Carica il mapping (oppure usa direttamente manual_merge se è già in memoria)
        # with open('/kaggle/working/gloss_merge_map.json', 'r', encoding='utf-8') as f:
        #     gloss_merge_map = json.load(f)

        # # 2. Crea una funzione che traduce una lista di glosse
        # def apply_gloss_merge(gloss_list, merge_map):
        #     """
        #     Itera su ogni glossa. Se è nel dizionario, la sostituisce.
        #     Altrimenti (usando .get), la lascia così com'è.
        #     """
        #     return [merge_map.get(gloss, gloss) for gloss in gloss_list]

In [8]:
# ── Diagnostica gloss e candidati merge ───────────────────────────────────
frequency_rows = [
    {
        'gloss': gloss,
        'count': count,
        'relative_freq': count / max(1, sum(merged_gloss_counter.values()))
    }
    for gloss, count in merged_gloss_counter.most_common()
]
frequency_df = pd.DataFrame(frequency_rows)
rare_glosses_df = frequency_df[frequency_df['count'] <= 2].copy()
rare_glosses_df = rare_glosses_df.sort_values(['count', 'gloss'], ascending=[True, True])

hyphen_tokens = [
    {'gloss': gloss, 'count': count, 'compact_form': gloss.replace('-', '')}
    for gloss, count in merged_gloss_counter.items()
    if '-' in gloss and gloss.replace('-', '') in merged_glosses and gloss.replace('-', '') != gloss
]
hyphen_df = pd.DataFrame(hyphen_tokens).sort_values(['count', 'gloss'], ascending=[False, True]) if hyphen_tokens else pd.DataFrame(columns=['gloss', 'count', 'compact_form'])

similar_candidates = []
top_tokens = [gloss for gloss, _ in merged_gloss_counter.most_common(250)]
for idx, gloss_a in enumerate(top_tokens):
    for gloss_b in top_tokens[idx + 1: idx + 21]:
        if gloss_a == gloss_b:
            continue
        if abs(len(gloss_a) - len(gloss_b)) > 3:
            continue
        ratio = SequenceMatcher(None, gloss_a, gloss_b).ratio()
        if ratio >= 0.88:
            similar_candidates.append({
                'gloss_a': gloss_a,
                'gloss_b': gloss_b,
                'similarity': round(ratio, 3),
                'count_a': int(merged_gloss_counter[gloss_a]),
                'count_b': int(merged_gloss_counter[gloss_b]),
            })

similar_df = pd.DataFrame(similar_candidates).sort_values(
    ['similarity', 'count_a', 'count_b'], ascending=[False, False, False]
) if similar_candidates else pd.DataFrame(columns=['gloss_a', 'gloss_b', 'similarity', 'count_a', 'count_b'])

class_reduction_report = {
    'raw_gloss_count': int(raw_gloss_count),
    'merged_gloss_count': int(merged_gloss_count),
    'class_reduction': int(merged_gloss_reduction),
    'raw_train_token_count': int(sum(raw_gloss_counter.values())),
    'merged_train_token_count': int(sum(merged_gloss_counter.values())),
    'rare_glosses_count': int(len(rare_glosses_df)),
    'hyphen_merge_candidates': int(len(hyphen_df)),
    'orthographic_similarity_candidates': int(len(similar_df)),
    'top_merged_glosses': [
        {'gloss': gloss, 'count': int(count)}
        for gloss, count in merged_gloss_counter.most_common(20)
    ],
}

if CONFIG['debug_save_reports']:
    frequency_df.to_csv(os.path.join(RESULTS_DIR, 'gloss_frequency_merged.csv'), index=False)
    rare_glosses_df.to_csv(os.path.join(RESULTS_DIR, 'gloss_rare_classes.csv'), index=False)
    hyphen_df.to_csv(os.path.join(RESULTS_DIR, 'gloss_hyphen_candidates.csv'), index=False)
    similar_df.to_csv(os.path.join(RESULTS_DIR, 'gloss_similarity_candidates.csv'), index=False)
    with open(os.path.join(RESULTS_DIR, 'gloss_class_diagnostics.json'), 'w', encoding='utf-8') as f:
        json.dump(class_reduction_report, f, indent=2, ensure_ascii=False)

print('\n=== Diagnostica gloss / classi ===')
print(f'Gloss raw unici:        {raw_gloss_count}')
print(f'Gloss dopo merge:       {merged_gloss_count}')
print(f'Riduzione classi:       {merged_gloss_reduction}')
print(f'Gloss rari (count <=2): {len(rare_glosses_df)}')
print(f'Candidati trattino:     {len(hyphen_df)}')
print(f'Candidati simili:       {len(similar_df)}')
print('\nTop 20 gloss dopo merge:')
for row in class_reduction_report['top_merged_glosses'][:20]:
    print(f"  {row['gloss']:<20} {row['count']}")

if len(hyphen_df) > 0:
    print('\nEsempi di merge ortografici sicuri:')
    for _, row in hyphen_df.head(15).iterrows():
        print(f"  {row['gloss']} -> {row['compact_form']}  (count={int(row['count'])})")

if len(similar_df) > 0:
    print('\nCoppie simili da valutare manualmente:')
    for _, row in similar_df.head(15).iterrows():
        print(
            f"  {row['gloss_a']} <-> {row['gloss_b']}  "
            f"(sim={row['similarity']}, counts={row['count_a']}/{row['count_b']})"
        )

print(f"\nReport salvati in: {RESULTS_DIR}")



=== Diagnostica gloss / classi ===
Gloss raw unici:        1115
Gloss dopo merge:       1115
Riduzione classi:       0
Gloss rari (count <=2): 484
Candidati trattino:     0
Candidati simili:       1

Top 20 gloss dopo merge:
  REGEN                2434
  REGION               2157
  IX                   1718
  MORGEN               1494
  KOMMEN               1483
  NORD                 1398
  SONNE                1294
  WOLKE                1268
  GRAD                 1218
  NACHT                982
  SUED                 949
  KOENNEN              933
  SCHNEE               888
  AUCH                 869
  MEHR                 850
  BISSCHEN             838
  HEUTE                835
  WETTER               801
  GEWITTER             779
  BIS                  768

Coppie simili da valutare manualmente:
  SCHOEN <-> SCHON  (sim=0.909, counts=212/168)

Report salvati in: /home/ebufi/phoenix/results


In [9]:
# ── Index keypoint files ─────────────────────────────────────────────────
def index_pose_files(split):
    split_dir = os.path.join(POSE_DIR, split)
    kp_dict = {}
    if not os.path.exists(split_dir):
        print(f'  ATTENZIONE: {split_dir} non trovata')
        return kp_dict
    for fname in os.listdir(split_dir):
        if fname.endswith('.npy'):
            kp_dict[os.path.splitext(fname)[0]] = os.path.join(split_dir, fname)
    return kp_dict

train_kp = index_pose_files('train')
dev_kp   = index_pose_files('dev')
test_kp  = index_pose_files('test')

print(f'Keypoints — Train: {len(train_kp)} | Dev: {len(dev_kp)} | Test: {len(test_kp)}')

Keypoints — Train: 2258 | Dev: 519 | Test: 642


## Step 3 — TSSI generation (versione corretta, senza duplicati)

In [ ]:

DFS_ORDER       = get_dfs_order_phoenix_correct()
EXPECTED_JOINTS = 48
DFS_UNIQUE      = list(dict.fromkeys(DFS_ORDER))   # joints unici in ordine DFS
print(f'✓ DFS_ORDER (versione corretta): {len(DFS_ORDER)} steps, '
      f'{len(DFS_UNIQUE)} joints unici')
 
# ============================================================================
# TSSI GENERATION V2  (Optimized with Confidence)
# ============================================================================
import numpy as np
from scipy.ndimage import gaussian_filter1d
import cv2
 
 
def generate_tssi_optimized(keypoints, frame_h=None):
    """
    Converte una sequenza di keypoints in una TSSI (5, n_joints, T).
 
    Args:
        keypoints : np.ndarray  shape (T, J, 2|3)
                    Canali: x, y  oppure  x, y, confidence.
        frame_h   : int | None
                    Se fornito, la dimensione temporale viene ridimensionata
                    a frame_h tramite resize bilineare (utile per batch
                    a lunghezza fissa).  Se None, la lunghezza originale
                    viene mantenuta e la sequenza è gestita dal collate_fn
                    con padding variabile (modalità CTC consigliata).
 
    Returns:
        tssi  : np.ndarray  shape (5, n_unique_joints, T_out)
        T_out : int         lunghezza temporale reale del TSSI restituito
    """
    try:
        kp    = keypoints.astype(np.float32)
        T_raw, J, C = kp.shape
 
        # ---- 0. Separa canali, gestisci assenza di confidence ----
        x    = kp[:, :, 0].copy()
        y    = kp[:, :, 1].copy()
        conf = kp[:, :, 2].copy() if C >= 3 else np.ones((T_raw, J), dtype=np.float32)
 
        # ---- 1. Interpolazione keypoint mancanti (conf < soglia) ----
        for j in range(J):
            valid_idx = np.where(conf[:, j] > 0.3)[0]
            if len(valid_idx) >= 2:
                t_all     = np.arange(T_raw)
                x[:, j]    = np.interp(t_all, valid_idx, x[valid_idx, j])
                y[:, j]    = np.interp(t_all, valid_idx, y[valid_idx, j])
                conf[:, j] = np.interp(t_all, valid_idx, conf[valid_idx, j])
            elif len(valid_idx) == 0:
                conf[:, j] = 0.0           # joint mai visibile → zero
 
        # ---- 2. Temporal smoothing (riduce jitter frame-by-frame) ----
        x    = gaussian_filter1d(x,    sigma=1.5, axis=0)
        y    = gaussian_filter1d(y,    sigma=1.5, axis=0)
        conf = gaussian_filter1d(conf, sigma=1.5, axis=0)

        # ---- 2b. Optical flow da keypoint (velocità) ----
        dx = np.zeros_like(x)
        dy = np.zeros_like(y)
        dx[1:] = x[1:] - x[:-1]
        dy[1:] = y[1:] - y[:-1]
        dx = gaussian_filter1d(dx, sigma=1.0, axis=0)
        dy = gaussian_filter1d(dy, sigma=1.0, axis=0)
        for arr in (dx, dy):
            std = arr.std() + 1e-6
            arr[:] = np.clip(arr / (3 * std), -1, 1)
 
        # ---- 3. Per-part normalization (body / face / hands separati) ----
        for start, end in [(0, 6), (6, 27), (27, 48)]:
            end = min(end, J)
            if end <= start:
                continue
            for arr in [x, y]:
                a_min = arr[:, start:end].min()
                a_max = arr[:, start:end].max()
                rng   = max(a_max - a_min, 1e-6)
                arr[:, start:end] = (arr[:, start:end] - a_min) / rng
 
        # ---- 4. Resize temporale (opzionale) ----
        if frame_h is not None:
            def _resize_temporal(arr_2d, target_h):
                """Resize (T, J) → (target_h, J) con bilinear."""
                if arr_2d.shape[0] == 1:
                    return np.repeat(arr_2d, target_h, axis=0)[:target_h]
                return cv2.resize(arr_2d.astype(np.float32),
                                  (J, target_h),
                                  interpolation=cv2.INTER_LINEAR)
 
            x    = _resize_temporal(x,    frame_h)
            y    = _resize_temporal(y,    frame_h)
            conf = _resize_temporal(conf, frame_h)
            dx   = _resize_temporal(dx,   frame_h)
            dy   = _resize_temporal(dy,   frame_h)
            T_out = frame_h
        else:
            T_out = T_raw
 
        # ---- 5. Riordina joints in DFS order ----
        dfs_unique = list(dict.fromkeys(DFS_ORDER))
        n_unique   = len(dfs_unique)
 
        # ---- 6. Costruisci TSSI  shape: (5, n_unique, T_out) ----
        tssi = np.zeros((5, n_unique, T_out), dtype=np.float32)
        for col_idx, joint_id in enumerate(dfs_unique):
            if joint_id >= J:
                continue
            tssi[0, col_idx, :] = np.clip(x[:,    joint_id], 0.0, 1.0)
            tssi[1, col_idx, :] = np.clip(y[:,    joint_id], 0.0, 1.0)
            tssi[2, col_idx, :] = np.clip(conf[:, joint_id], 0.0, 1.0)
            tssi[3, col_idx, :] = dx[:, joint_id]
            tssi[4, col_idx, :] = dy[:, joint_id]
 
        return tssi, T_out
 
    except Exception as e:
        print(f'[generate_tssi_optimized] Errore: {e}')
        import traceback; traceback.print_exc()
        fallback_h = frame_h if frame_h is not None else (T_raw if 'T_raw' in dir() else 1)
        expected_j = globals().get('EXPECTED_JOINTS', len(list(dict.fromkeys(DFS_ORDER))))
        return np.zeros((5, expected_j, fallback_h), dtype=np.float32), 1
 
 
print('✓ generate_tssi_optimized definita correttamente')

✓ DFS_ORDER (versione corretta): 135 steps, 48 joints unici
✓ generate_tssi_optimized definita correttamente


In [11]:
import os
import numpy as np
from tqdm import tqdm # Aggiungiamo tqdm per avere una bella barra di caricamento!

# 1. Crea le cartelle fisiche se non esistono
for split in ['train', 'dev', 'test']:
    os.makedirs(os.path.join(TSSI_OUTPUT_DIR, split), exist_ok=True)

print("Inizio generazione cache TSSI (formato .npz)...")

# 2. Genera e salva i TSSI per il TRAIN
print("\nElaborazione Train...")
for vid_name, kp_path in tqdm(train_kp.items(), desc="Train"):
    save_path = os.path.join(TSSI_OUTPUT_DIR, 'train', f"{vid_name}.npz") # Usiamo .npz
    if not os.path.exists(save_path):
        # CARICHIAMO IL FILE DAL DISCO ALLA RAM
        kp_array = np.load(kp_path) 
        
        # Generiamo il TSSI (che restituisce la matrice e la lunghezza)
        tssi_tensor, seq_len = generate_tssi_optimized(kp_array) 
        
        # Salviamo come archivio compresso npz (esattamente come lo vuole il dataset!)
        np.savez_compressed(save_path, tssi=tssi_tensor, seq_len=seq_len)

# 3. Genera e salva i TSSI per il DEV
print("\nElaborazione Dev...")
for vid_name, kp_path in tqdm(dev_kp.items(), desc="Dev"):
    save_path = os.path.join(TSSI_OUTPUT_DIR, 'dev', f"{vid_name}.npz")
    if not os.path.exists(save_path):
        kp_array = np.load(kp_path)
        tssi_tensor, seq_len = generate_tssi_optimized(kp_array)
        np.savez_compressed(save_path, tssi=tssi_tensor, seq_len=seq_len)

# 4. Genera e salva i TSSI per il TEST
print("\nElaborazione Test...")
for vid_name, kp_path in tqdm(test_kp.items(), desc="Test"):
    save_path = os.path.join(TSSI_OUTPUT_DIR, 'test', f"{vid_name}.npz")
    if not os.path.exists(save_path):
        kp_array = np.load(kp_path)
        tssi_tensor, seq_len = generate_tssi_optimized(kp_array)
        np.savez_compressed(save_path, tssi=tssi_tensor, seq_len=seq_len)

print("\nGenerazione TSSI completata! Ora puoi creare i Dataset.")

Inizio generazione cache TSSI (formato .npz)...

Elaborazione Train...


Train: 100%|██████████| 2258/2258 [00:00<00:00, 341497.08it/s]



Elaborazione Dev...


Dev: 100%|██████████| 519/519 [00:00<00:00, 286012.85it/s]



Elaborazione Test...


Test: 100%|██████████| 642/642 [00:00<00:00, 292530.49it/s]


Generazione TSSI completata! Ora puoi creare i Dataset.


## Step 4 — Dataset e DataLoader

In [12]:
# ============================================================================
# STEP 1: MERGE GLOSS SIMILI (e.g., SCHOEN → SCHON)
# ============================================================================

from difflib import SequenceMatcher

def build_merge_map(glosses, similarity_threshold=0.85):
    """Crea mappa di merge per gloss simili."""
    glosses_sorted = sorted(glosses)
    merge_map = {}
    
    for i, g1 in enumerate(glosses_sorted):
        if g1 in merge_map:
            continue
        for g2 in glosses_sorted[i+1:]:
            if g2 in merge_map:
                continue
            ratio = SequenceMatcher(None, g1, g2).ratio()
            if ratio >= similarity_threshold:
                merge_map[g2] = g1
    
    return merge_map

def apply_merge_to_df(df, merge_map):
    """Applica il merge map a un dataframe."""
    df = df.copy()
    new_orth = []
    for gloss_seq in df['orth'].fillna(''):
        tokens = str(gloss_seq).strip().upper().split()
        merged_tokens = [merge_map.get(t, t) for t in tokens]
        new_orth.append(' '.join(merged_tokens))
    df['orth'] = new_orth
    return df

# Raccogli gloss per creare mappa merge
all_glosses_raw = set()
for df in [train_df, dev_df, test_df]:
    for gloss_seq in df['orth'].fillna(''):
        all_glosses_raw.update(str(gloss_seq).strip().upper().split())

merge_map = build_merge_map(all_glosses_raw, similarity_threshold=0.85)

train_df_merged = apply_merge_to_df(train_df, merge_map)
dev_df_merged   = apply_merge_to_df(dev_df, merge_map)
test_df_merged  = apply_merge_to_df(test_df, merge_map)

print(f'✓ Merge gloss simili completato: {sum(1 for v in merge_map.values() if v)} gloss mergiate')

# ============================================================================
# STEP 2: MAPPING UNKNOWN PER GLOSS NON-TRAIN
# ============================================================================

def collect_gloss_set(df):
    gloss_set = set()
    for gloss_seq in df['orth'].fillna(''):
        gloss_set.update(str(gloss_seq).strip().upper().split())
    return gloss_set

train_glosses = collect_gloss_set(train_df_merged)
dev_glosses   = collect_gloss_set(dev_df_merged)
test_glosses  = collect_gloss_set(test_df_merged)

unknown_gloss_map = {}
for gloss in (dev_glosses | test_glosses) - train_glosses:
    unknown_gloss_map[gloss] = '<unk>'

def apply_unknown_to_df(df, unknown_map):
    df = df.copy()
    new_orth = []
    for gloss_seq in df['orth'].fillna(''):
        tokens = str(gloss_seq).strip().upper().split()
        mapped_tokens = [unknown_map.get(t, t) for t in tokens]
        new_orth.append(' '.join(mapped_tokens))
    df['orth'] = new_orth
    return df

dev_df_merged = apply_unknown_to_df(dev_df_merged, unknown_gloss_map)
test_df_merged = apply_unknown_to_df(test_df_merged, unknown_gloss_map)

print(f'✓ Mapping unknown completato: {len(unknown_gloss_map)} gloss → <unk>')

# ============================================================================
# STEP 3: VOCABOLARIO CON <blank>, <unk> E GLOSS TRAIN
# ============================================================================

vocab = ['<blank>', '<unk>'] + sorted(train_glosses)
g2i = {g: i for i, g in enumerate(vocab)}
i2g = {i: g for g, i in g2i.items()}
num_classes = len(vocab)

print(f"✓ Vocabolario definitivo:")
print(f"  <blank>: ID={g2i['<blank>']}")
print(f"  <unk>:   ID={g2i['<unk>']}")
print(f"  Gloss train: {len(train_glosses)}")
print(f"  Totale classi: {num_classes}")

# Verifica
print(f"\nVerifica:")
print(f"  g2i['<blank>'] = {g2i.get('<blank>', 'MANCA!')}")
print(f"  g2i['<unk>'] = {g2i.get('<unk>', 'MANCA!')}")
print(f"  i2g[0] = {i2g.get(0, 'MANCA!')}")
print(f"  i2g[1] = {i2g.get(1, 'MANCA!')}")

# ============================================================================
# STEP 4: CALCOLO LOG_PRIOR CON LE NUOVE DIMENSIONI
# ============================================================================

token_counts = Counter()
for seq in train_df_merged['orth'].fillna(''):
    tokens = str(seq).strip().upper().split()
    for token in tokens:
        if token in g2i:
            token_counts[g2i[token]] += 1

prior = torch.zeros(num_classes)
for idx, cnt in token_counts.items():
    prior[idx] = cnt
prior[0] = prior[1:].sum() * 0.01  # prior blank basso ma > 0
prior = prior / prior.sum()
LOG_PRIOR = torch.log(prior + 1e-8)  # shape (num_classes,)

print(f'✓ Log-prior ricalcolato con {num_classes} classi (β={CONFIG["prior_beta"]})')

class PHOENIXDatasetContinuos(Dataset):
    def __init__(self, df, kp_dict, g2i, augment=False,
                 frame_h=CONFIG['frame_h'], tssi_dir=None):
        self.kp_dict  = kp_dict
        self.g2i      = g2i
        self.augment  = augment
        self.frame_h  = frame_h
        self.tssi_dir = tssi_dir  # se non None, carica npz precomputati
        self.samples  = []

        for _, row in df.iterrows():
            vid = str(row['name'])
            if vid not in kp_dict:
                continue
            gloss_str = str(row['orth']).strip().upper()
            labels = [g2i[g] for g in gloss_str.split() if g in g2i]
            if labels:
                self.samples.append({'vid': vid, 'labels': labels})

        print(f'Dataset: {len(self.samples)} samples | augment={augment}')

    def __len__(self):
        return len(self.samples)

    def _augment_tssi(self, tssi):
        # tssi: (3, J, T) — canali x, y, conf
        C, J, T = tssi.shape

        # ---- 1. Jitter gaussiano correlato nel tempo ----
        if np.random.rand() < 0.8:
            from scipy.ndimage import gaussian_filter1d
            noise = np.random.randn(2, J, T).astype(np.float32)
            noise = gaussian_filter1d(noise, sigma=3, axis=2)
            noise *= 0.008
            tssi[:2] = np.clip(tssi[:2] + noise, 0, 1)

        # ---- 2. Temporal warping ----
        if np.random.rand() < 0.7:
            speeds = np.random.uniform(0.7, 1.3, T).astype(np.float32)
            idx = np.cumsum(speeds)
            idx = (idx - idx[0]) / (idx[-1] - idx[0]) * (T - 1)
            warped = np.zeros_like(tssi)
            for c in range(C):
                for j in range(J):
                    warped[c, j, :] = np.interp(
                        np.arange(T), idx, tssi[c, j, :])
            tssi = warped

        # ---- 3. Flip orizzontale ----
        # Mappa speculare: scambia mano sinistra (joint 6-26) con destra (27-47)
        if np.random.rand() < 0.5:
            tssi = tssi.copy()
            tssi[0] = 1.0 - tssi[0]  # ribalta coordinata x
            # scambia i joint delle due mani
            left  = list(range(6, 27))
            right = list(range(27, 48))
            if J >= 48:
                tssi[:, left + right, :] = tssi[:, right + left, :]

        # ---- 4. Scaling casuale ----
        if np.random.rand() < 0.6:
            scale = np.random.uniform(0.85, 1.15)
            tssi[:2] = np.clip(tssi[:2] * scale, 0, 1)

        # ---- 5. Frame dropout ----
        if np.random.rand() < 0.5:
            n_drop = np.random.randint(1, max(2, int(T * 0.05)))
            drop_idx = np.random.choice(T, n_drop, replace=False)
            for i in sorted(drop_idx):
                if i > 0:
                    tssi[:, :, i] = tssi[:, :, i - 1]

        # ---- 6. Confidence dropout ----
        if np.random.rand() < 0.4:
            mask = np.random.rand(J, T) < 0.08
            tssi[2, mask] = 0.0

        return tssi

    def __getitem__(self, idx):
        s   = self.samples[idx]
        vid = s['vid']

        # Carica da npz precomputato se disponibile
        if self.tssi_dir is not None:
            npz_path = os.path.join(self.tssi_dir, f'{vid}.npz')
            if os.path.exists(npz_path):
                data    = np.load(npz_path, allow_pickle=False)
                tssi    = data['tssi'].astype(np.float32)
                seq_len = int(data['seq_len'])
            else:
                kp   = np.load(self.kp_dict[vid]).astype(np.float32)
                tssi, seq_len = generate_tssi_optimized(kp, frame_h=None)
        else:
            kp   = np.load(self.kp_dict[vid]).astype(np.float32)
            tssi, seq_len = generate_tssi_optimized(kp, frame_h=None)

        if self.augment:
            tssi = self._augment_tssi(tssi)
        return {
            'tssi':    torch.from_numpy(tssi).float(),
            'labels':  s['labels'],
            'seq_len': seq_len,
        }
def collate_fn_ctc(batch):
    max_h = max(b['tssi'].shape[2] for b in batch)
    tssies, input_lengths = [], []
    for b in batch:
        t = b['tssi']
        pad = max_h - t.shape[2]
        tssies.append(F.pad(t, (0, pad)))
        input_lengths.append(t.shape[2])
    tssies         = torch.stack(tssies)
    targets        = torch.cat([torch.LongTensor(b['labels']) for b in batch])
    input_lengths  = torch.LongTensor(input_lengths)
    target_lengths = torch.LongTensor([len(b['labels']) for b in batch])
    return tssies, targets, input_lengths, target_lengths
# Crea dataset e dataloader
TSSI_OUTPUT_DIR = '/kaggle/working/'

# Usa i dataframe con merge e unknown mapping applicati
train_ds = PHOENIXDatasetContinuos(train_df_merged, train_kp, g2i,
                               augment=True,  tssi_dir=os.path.join(TSSI_OUTPUT_DIR, 'train'))
dev_ds   = PHOENIXDatasetContinuos(dev_df_merged,   dev_kp,   g2i,
                               augment=False, tssi_dir=os.path.join(TSSI_OUTPUT_DIR, 'dev'))
test_ds  = PHOENIXDatasetContinuos(test_df_merged,  test_kp,  g2i,
                               augment=False, tssi_dir=os.path.join(TSSI_OUTPUT_DIR, 'test'))

BS          = CONFIG['batch_size']   # 16
NUM_WORKERS = 4

train_dl = DataLoader(train_ds, batch_size=BS, shuffle=True,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      collate_fn=collate_fn_ctc, drop_last=True)
dev_dl   = DataLoader(dev_ds,   batch_size=BS, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      collate_fn=collate_fn_ctc)
test_dl  = DataLoader(test_ds,  batch_size=BS, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      collate_fn=collate_fn_ctc)

print(f'BS={BS} | Train={len(train_dl)} | Dev={len(dev_dl)} | Test={len(test_dl)}')

✓ Merge gloss simili completato: 68 gloss mergiate
✓ Mapping unknown completato: 27 gloss → <unk>
✓ Vocabolario definitivo:
  <blank>: ID=0
  <unk>:   ID=1
  Gloss train: 1020
  Totale classi: 1022

Verifica:
  g2i['<blank>'] = 0
  g2i['<unk>'] = 1
  i2g[0] = <blank>
  i2g[1] = <unk>
✓ Log-prior ricalcolato con 1022 classi (β=0.32404536014294083)
Dataset: 2258 samples | augment=True
Dataset: 519 samples | augment=False
Dataset: 642 samples | augment=False
BS=16 | Train=141 | Dev=33 | Test=41


## Step 5 — Modello con Gradient Checkpointing (fix OOM)

In [13]:
class CTCLossWithSmoothing(nn.Module):
    """Mantienuta per compatibilità ma utilizza CTCLossWithEntropy internamente."""
    def __init__(self, blank=0, smoothing=0.05):
        super().__init__()
        self.smoothing = smoothing
        # Usa la versione ottimizzata con entropy regularization
        self.ctc = CTCLossWithEntropy(blank=blank, entropy_weight=smoothing)

    def forward(self, log_probs_TBC, targets, input_lengths, target_lengths):
        return self.ctc(log_probs_TBC, targets, input_lengths, target_lengths)




print("✓ Metriche epoch aggregate definite")


✓ Metriche epoch aggregate definite


## Step 6 — Two-Phase Finetuning (Deep Sign §6.2)

In [ ]:
def freeze_backbone(model):
    # Congela backbone e attenzione temporale
    for module in [model.gcn_backbone, model.temporal_attn]:
        for p in module.parameters():
            p.requires_grad = False
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Backbone congelato — parametri trainable: {n:,}')


def unfreeze_backbone(model):
    for p in model.parameters():
        p.requires_grad = True
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Backbone sbloccato — parametri trainable: {n:,}')

def make_optimizer_phase1(model, lr, weight_decay):
    """Fase 1: solo BiLSTM + norm + FC."""
    params = [p for p in model.parameters() if p.requires_grad]
    return torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)


def make_optimizer_phase2(model, lr_backbone, lr_head, weight_decay):
    """Fase 2: backbone lento, testa veloce."""
    return torch.optim.AdamW([
        {'params': model.gcn_backbone.parameters(),  'lr': lr_backbone},
        {'params': model.temporal_attn.parameters(), 'lr': lr_backbone},
        {'params': model.bilstm.parameters(),        'lr': lr_head},
        {'params': model.norm.parameters(),          'lr': lr_head},
        {'params': model.fc.parameters(),            'lr': lr_head},
        {'params': model.aux_proj.parameters(),      'lr': lr_head},
    ], weight_decay=weight_decay)

def make_scheduler(optimizer, num_epochs, steps_per_epoch):
    from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
    warmup_steps = 5 * steps_per_epoch
    total_steps  = num_epochs * steps_per_epoch
    warmup = LinearLR(optimizer, start_factor=0.1, total_iters=warmup_steps)
    cosine = CosineAnnealingLR(optimizer, T_max=total_steps - warmup_steps, eta_min=1e-6)
    return SequentialLR(optimizer, schedulers=[warmup, cosine],
                        milestones=[warmup_steps])


print('✓ Funzioni two-phase finetuning aggiornate per TCN')

✓ Funzioni two-phase finetuning aggiornate per TCN


## Step 7 — Decoding con Prior Scaling (Deep Sign Eq.13)

In [15]:
def collapse_ctc(token_ids):
    """Rimuove blank (0) e duplicati consecutivi."""
    result, prev = [], None
    for idx in token_ids:
        v = idx.item() if hasattr(idx, 'item') else idx
        if v != 0 and v != prev:
            result.append(v)
        prev = v
    return result


def beam_ctc_decode_with_prior(log_probs_TBC, beam_width=10, beta=0.0, log_prior=None):
    """
    Beam search CTC con prior scaling (Deep Sign Eq.13).
    Usa la versione ottimizzata da optimized_code.py.
    """
    # Usa la versione ottimizzata
    return beam_search_ctc_optimized(
        log_probs_TBC=log_probs_TBC,
        beam_width=beam_width,
        beta=beta,
        log_prior=log_prior
    )


def greedy_decode_with_prior(log_probs_TBC, beta=0.0, log_prior=None):
    """Greedy decode con prior scaling — più veloce del beam, utile durante il training."""
    T, B, C = log_probs_TBC.shape
    results = []
    for b in range(B):
        lp = log_probs_TBC[:, b, :]
        if beta > 0.0 and log_prior is not None:
            lp = lp - beta * log_prior.unsqueeze(0)
        best_ids = lp.argmax(dim=-1).tolist()
        results.append(collapse_ctc(best_ids))
    return results

def compute_word_recognition_metrics(all_refs, all_hyps, i2g, min_support=5,
                                     include_unk=False):
    """
    Calcola precision, recall e F1 per ogni gloss individuale.

    Strategia: per ogni gloss k, conta quante volte appare in REF vs HYP,
    usando il multiset-overlap (intersection of bags) come true positive.
    Questo è identico alla logica del SacreBLEU per i conteggi.

    Parameters
    ----------
    include_unk : bool
        Se True, include <unk> (id=1) come classe valida da riconoscere,
        cioè una predizione <unk> su un token <unk> è contata come TP.
        Se False (default), <unk> è esclusa dalle metriche per classe.

    Returns
    -------
    summary : dict con macro_f1, macro_precision, macro_recall
    per_class : list of dicts (gloss, precision, recall, f1, support)
    low_recall : list di glosse con recall < 0.2 e support >= min_support
    never_predicted : list di glosse mai predette (recall = 0)
    """
    # Accumula conteggi per classe
    ref_count  = defaultdict(int)   # quante volte gloss k appare nelle REF
    hyp_count  = defaultdict(int)   # quante volte appare nelle HYP
    tp_count   = defaultdict(int)   # veri positivi (overlap bag)

    for ref, hyp in zip(all_refs, all_hyps):
        # Multiset overlap: min(count_ref, count_hyp) per ogni token
        ref_bag = defaultdict(int)
        hyp_bag = defaultdict(int)
        for r in ref: ref_bag[r] += 1
        for h in hyp: hyp_bag[h] += 1

        all_keys = set(ref_bag) | set(hyp_bag)
        for k in all_keys:
            ref_count[k] += ref_bag[k]
            hyp_count[k] += hyp_bag[k]
            tp_count[k]  += min(ref_bag[k], hyp_bag[k])

    # Escludi sempre blank (id=0).
    # <unk> (id=1): escluso di default, incluso se include_unk=True.
    min_id = 1 if include_unk else 2
    gloss_ids = [k for k in ref_count if k >= min_id]

    per_class = []
    for k in gloss_ids:
        support = ref_count[k]
        tp = tp_count[k]
        precision = tp / hyp_count[k] if hyp_count[k] > 0 else 0.0
        recall    = tp / ref_count[k] if ref_count[k] > 0 else 0.0
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        per_class.append({
            'id':        k,
            'gloss':     i2g.get(k, f'[{k}]'),
            'precision': round(precision, 4),
            'recall':    round(recall, 4),
            'f1':        round(f1, 4),
            'support':   support,
            'tp':        tp,
            'predicted': hyp_count[k],
        })

    # Ordina per support decrescente
    per_class.sort(key=lambda x: -x['support'])

    # Macro F1 pesato per support (weighted macro)
    total_support = sum(d['support'] for d in per_class)
    if total_support > 0:
        macro_f1        = sum(d['f1']        * d['support'] for d in per_class) / total_support
        macro_precision = sum(d['precision'] * d['support'] for d in per_class) / total_support
        macro_recall    = sum(d['recall']    * d['support'] for d in per_class) / total_support
    else:
        macro_f1 = macro_precision = macro_recall = 0.0

    # Classi mai predette (recall=0) con support sufficiente
    never_predicted = [d for d in per_class if d['predicted'] == 0 and d['support'] >= min_support]
    low_recall      = [d for d in per_class if 0 < d['recall'] < 0.20 and d['support'] >= min_support]

    summary = {
        'macro_f1':        round(macro_f1, 4),
        'macro_precision': round(macro_precision, 4),
        'macro_recall':    round(macro_recall, 4),
        'n_classes':       len(per_class),
        'n_never_pred':    len(never_predicted),
        'n_low_recall':    len(low_recall),
        'include_unk':     include_unk,
    }
    return summary, per_class, never_predicted, low_recall
 
 
def print_word_metrics(summary, per_class, never_predicted, low_recall,
                       summary_unk=None, per_class_unk=None,
                       never_predicted_unk=None, low_recall_unk=None,
                       top_k=15, epoch=None, split='VAL'):
    """Stampa un report leggibile delle metriche per singola parola.

    Se summary_unk è fornito (metriche con <unk> trattata come classe valida),
    viene stampata una seconda sezione di confronto.
    """
    ep_str = f' Ep {epoch}' if epoch is not None else ''

    # ── Sezione standard (unk esclusa) ──────────────────────────────────
    print(f'\n{"─"*65}')
    print(f'[WORD METRICS {split}{ep_str}]  (escluso <unk>)')
    print(f'  Macro-F1:        {summary["macro_f1"]*100:.2f}%')
    print(f'  Macro-Precision: {summary["macro_precision"]*100:.2f}%')
    print(f'  Macro-Recall:    {summary["macro_recall"]*100:.2f}%')
    print(f'  Classi totali:   {summary["n_classes"]}')
    print(f'  Mai predette:    {summary["n_never_pred"]}  (support ≥ 5)')
    print(f'  Low recall<20%:  {summary["n_low_recall"]}  (support ≥ 5)')

    print(f'\n  Top-{top_k} classi per support:')
    print(f'  {"Gloss":<22} {"Sup":>5} {"Prec":>6} {"Rec":>6} {"F1":>6}')
    print(f'  {"─"*50}')
    for d in per_class[:top_k]:
        print(f'  {d["gloss"]:<22} {d["support"]:>5} '
              f'{d["precision"]*100:>5.1f}% {d["recall"]*100:>5.1f}% {d["f1"]*100:>5.1f}%')

    if never_predicted:
        print(f'\n  ⚠ Classi frequenti MAI predette (top-10):')
        for d in sorted(never_predicted, key=lambda x: -x['support'])[:10]:
            print(f'    {d["gloss"]:<22} support={d["support"]}')

    # ── Sezione con <unk> come classe valida ────────────────────────────
    if summary_unk is not None:
        print(f'\n{"─"*65}')
        print(f'[WORD METRICS {split}{ep_str}]  (incluso <unk> come classe)')
        print(f'  Macro-F1:        {summary_unk["macro_f1"]*100:.2f}%')
        print(f'  Macro-Precision: {summary_unk["macro_precision"]*100:.2f}%')
        print(f'  Macro-Recall:    {summary_unk["macro_recall"]*100:.2f}%')
        print(f'  Classi totali:   {summary_unk["n_classes"]}  (include <unk>)')
        print(f'  Mai predette:    {summary_unk["n_never_pred"]}  (support ≥ 5)')
        print(f'  Low recall<20%:  {summary_unk["n_low_recall"]}  (support ≥ 5)')

        # Differenza rispetto alla versione senza unk
        delta_f1 = (summary_unk["macro_f1"] - summary["macro_f1"]) * 100
        delta_rec = (summary_unk["macro_recall"] - summary["macro_recall"]) * 100
        sign_f1  = "+" if delta_f1  >= 0 else ""
        sign_rec = "+" if delta_rec >= 0 else ""
        print(f'  Δ Macro-F1 vs no-unk: {sign_f1}{delta_f1:.2f}pp')
        print(f'  Δ Macro-Recall:       {sign_rec}{delta_rec:.2f}pp')

        if per_class_unk:
            # Mostra riga <unk> per evidenziarla
            unk_rows = [d for d in per_class_unk if d['gloss'] == '<unk>']
            if unk_rows:
                d = unk_rows[0]
                print(f'\n  Riga <unk>: support={d["support"]} | '
                      f'Prec={d["precision"]*100:.1f}% | '
                      f'Rec={d["recall"]*100:.1f}% | '
                      f'F1={d["f1"]*100:.1f}%  (pred={d["predicted"]}, tp={d["tp"]})')

        if never_predicted_unk:
            print(f'\n  ⚠ Classi MAI predette incl. <unk> (top-10):')
            for d in sorted(never_predicted_unk, key=lambda x: -x['support'])[:10]:
                print(f'    {d["gloss"]:<22} support={d["support"]}')

    print(f'{"─"*65}')
 

def compute_wer(refs, hyps):
    """
    Calcola Word Error Rate e restituisce il breakdown esatto:
    (WER, Sostituzioni, Cancellazioni, Inserzioni, Totale Parole)
    """
    total_s, total_d, total_i, total_n = 0, 0, 0, 0
    
    for ref, hyp in zip(refs, hyps):
        r, h = len(ref), len(hyp)
        total_n += r
        
        if r == 0:
            total_i += h
            continue
            
        # 1. Inizializzazione Matrice DP
        dp = [[0] * (h + 1) for _ in range(r + 1)]
        for i in range(1, r + 1): dp[i][0] = i
        for j in range(1, h + 1): dp[0][j] = j
        
        # 2. Forward pass (Calcolo distanze)
        for i in range(1, r + 1):
            for j in range(1, h + 1):
                if ref[i-1] == hyp[j-1]:
                    dp[i][j] = dp[i-1][j-1]
                else:
                    sub = dp[i-1][j-1] + 1
                    dlt = dp[i-1][j] + 1
                    ins = dp[i][j-1] + 1
                    dp[i][j] = min(sub, dlt, ins)
                    
        # 3. Backtracking (Isolamento degli errori)
        i, j = r, h
        s, d, ins = 0, 0, 0
        while i > 0 or j > 0:
            if i > 0 and j > 0 and ref[i-1] == hyp[j-1]:
                i -= 1; j -= 1
            elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + 1:
                s += 1
                i -= 1; j -= 1
            elif i > 0 and dp[i][j] == dp[i-1][j] + 1:
                d += 1
                i -= 1
            else:
                ins += 1
                j -= 1
                
        total_s += s
        total_d += d
        total_i += ins

    # Evitiamo divisioni per zero
    N = max(total_n, 1)
    wer = (total_s + total_d + total_i) / N
    
    # Restituiamo un dizionario per comodità
    return wer, {'S': total_s, 'D': total_d, 'I': total_i, 'N': N}

print('✓ Decoding con prior scaling definito (versione ottimizzata)')

✓ Decoding con prior scaling definito (versione ottimizzata)


## Step 8 — Training loop

In [ ]:
def ids_to_gloss_text(ids, i2g):
    tokens = []
    for idx in ids:
        token_id = int(idx.item() if hasattr(idx, 'item') else idx)
        if token_id == 0:
            continue
        tokens.append(i2g.get(token_id, f'<unk:{token_id}>'))
    return ' '.join(tokens) if tokens else '<blank>'


def format_counter(counter, i2g, top_k=8):
    if not counter:
        return '<vuoto>'
    return ', '.join(
        f'{i2g.get(idx, f"<unk:{idx}>")}:{count}'
        for idx, count in counter.most_common(top_k)
    )


# ============================================================================
# RUN_EPOCH PULITA - Raccoglie metriche, stampa UNA VOLTA alla fine
# ============================================================================
def run_epoch(model, dl, criterion, optimizer, scaler, scheduler,
              training, device, config, log_prior, i2g):
    """
    Training/validation epoch.
    Gestisce auxiliary CTC loss (Fix 4) quando model.training restituisce
    una tupla (main_log_prob, aux_log_prob).
    """
    model.train() if training else model.eval()
    total_loss = 0.0
    all_refs, all_hyps = [], []
    all_pred_counter = Counter()
    all_ref_counter  = Counter()

    accum_steps = config['gradient_accumulation_steps']
    beta        = config['prior_beta']
    grad_clip   = config.get('grad_clip', 5.0)

    if training:
        optimizer.zero_grad(set_to_none=True)

    for batch_idx, (tssies, targets, input_lengths, target_lengths) in enumerate(
            tqdm(dl, leave=False, desc='Train' if training else 'Val')):

        tssies = tssies.to(device, non_blocking=True)
        do_mixup = training and np.random.rand() < 0.4
        if do_mixup:
            lam     = float(np.random.beta(0.3, 0.3))
            mix_idx = torch.randperm(tssies.shape[0], device=device)
            tssies  = lam * tssies + (1 - lam) * tssies[mix_idx]
        with torch.set_grad_enabled(training):
            with torch.autocast(device_type=device.type, dtype=torch.float16,
                                 enabled=config['use_amp'] and device.type == 'cuda'):

                # ── Forward — gestisce modello con o senza auxiliary head ──
                output = model(tssies)
                if isinstance(output, tuple):
                    # Fix 4 attivo: (main_log_prob, aux_log_prob)
                    main_lp, aux_lp = output
                    has_aux = True
                else:
                    main_lp = output
                    has_aux = False

                # (B, T, C) → (T, B, C) per CTC
                lp_float = main_lp.permute(1, 0, 2).float()
                T_out    = lp_float.shape[0]
                T_in     = tssies.shape[3]
                scale    = T_out / max(T_in, 1)

                ctc_input_lengths  = (input_lengths.float() * scale).long().clamp(1, T_out).cpu()
                ctc_target_lengths = target_lengths.to(dtype=torch.long, device='cpu')
                ctc_targets        = targets.to(dtype=torch.long, device='cpu')

                loss_main = criterion(lp_float, ctc_targets,
                                      ctc_input_lengths, ctc_target_lengths)

                if training and do_mixup:
                    target_list            = torch.split(ctc_targets, ctc_target_lengths.tolist())
                    target_list_mix        = [target_list[i] for i in mix_idx.cpu().tolist()]
                    ctc_targets_mix        = torch.cat(target_list_mix)
                    ctc_target_lengths_mix = ctc_target_lengths[mix_idx.cpu()]

                    loss_mix  = criterion(lp_float, ctc_targets_mix,
                                          ctc_input_lengths, ctc_target_lengths_mix)
                    loss_main = lam * loss_main + (1 - lam) * loss_mix

                if training and has_aux:
                    aux_float = aux_lp.permute(1, 0, 2).float()
                    loss_aux  = criterion(aux_float, ctc_targets,
                                          ctc_input_lengths, ctc_target_lengths)
                    loss = 0.7 * loss_main + 0.3 * loss_aux
                else:
                    loss = loss_main

            if training:
                scaler.scale(loss / accum_steps).backward()
                if (batch_idx + 1) % accum_steps == 0 or \
                   (batch_idx + 1) == len(dl):
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    if scheduler is not None:
                        scheduler.step()

        total_loss += loss.item()

        # ── Decode ────────────────────────────────────────────────────────
        with torch.no_grad():
            lp_cpu = lp_float.detach().cpu()
            if training:
                decoded = greedy_decode_with_prior(lp_cpu, beta=0.0)
            else:
                decoded = greedy_decode_with_prior(lp_cpu, beta=beta,
                                                   log_prior=log_prior.cpu())

        # ── Accumula refs ─────────────────────────────────────────────────
        refs, offset = [], 0
        for tlen in ctc_target_lengths.tolist():
            refs.append(ctc_targets[offset:offset + tlen].tolist())
            offset += tlen

        all_refs.extend(refs)
        all_hyps.extend(decoded)
        for ref in refs:
            all_ref_counter.update(ref)
        for hyp in decoded:
            all_pred_counter.update(hyp)

        del tssies, lp_float, lp_cpu, loss

    # ── Metriche finali ───────────────────────────────────────────────────
    avg_loss           = total_loss / len(dl)
    avg_wer, wer_details = compute_wer(all_refs, all_hyps)

    metrics = compute_epoch_metrics(all_refs, all_hyps, all_pred_counter,
                                    all_ref_counter, i2g, top_k=8)
    metrics['wer_details'] = wer_details

    word_summary, per_class, never_pred, low_rec = compute_word_recognition_metrics(
        all_refs, all_hyps, i2g, include_unk=False)

    word_summary_unk, per_class_unk, never_pred_unk, low_rec_unk = compute_word_recognition_metrics(
        all_refs, all_hyps, i2g, include_unk=True)

    return (avg_loss, avg_wer, metrics,
            word_summary, per_class, never_pred, low_rec,
            word_summary_unk, per_class_unk, never_pred_unk, low_rec_unk)


print('✓ run_epoch definita e pronta all\'uso')

  



print('✓ run_epoch_clean() definita e pronta all\'uso')


# ============================================================================
# TCN-BiLSTM-CTC Model (Pose Network Approach - STMC Paper)
# ============================================================================

class TCNBlock(nn.Module):
    """
    Residual TCN block con dilated convolution + DropPath.
    DropPath viene applicato all'output del ramo conv (non al residual):
    in questo modo il gradiente scorre sempre attraverso il residual
    anche quando il blocco viene "saltato".
    """
    def __init__(self, in_channels, out_channels,
                 kernel_size=3, dilation=1,
                 dropout=0.2, drop_path_rate=0.0):   # ← nuovo parametro
        super().__init__()
        pad = dilation * (kernel_size - 1) // 2
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size,
                      padding=pad, dilation=dilation),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Conv1d(out_channels, out_channels, kernel_size,
                      padding=pad, dilation=dilation),
            nn.BatchNorm1d(out_channels),
        )
        self.skip = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=1),
            nn.BatchNorm1d(out_channels),
        ) if in_channels != out_channels else nn.Identity()

        self.drop_path = DropPath(drop_path_rate)   # ← nuovo

    def forward(self, x):
        residual = self.skip(x)
        # DropPath sul ramo conv, residual sempre attivo
        return F.relu(self.drop_path(self.net(x)) + residual)   # ← modificatolu(self.net(x) + residual)


class PoseNetworkTCN(nn.Module):
    """TCN con DropPath lineare crescente sui blocchi."""
    def __init__(self, num_joints=48, num_channels=3, hidden_dim=256,
                 num_blocks=4, dropout=0.3,
                 drop_path_rate=0.1):   # ← nuovo parametro
        super().__init__()
        self.num_joints  = num_joints
        self.in_channels = num_joints * num_channels

        # Drop path cresce linearmente: blocco 0 → 0, blocco N-1 → drop_path_rate
        dp_rates = [
            drop_path_rate * i / max(num_blocks - 1, 1)
            for i in range(num_blocks)
        ]

        layers = [
            TCNBlock(self.in_channels, hidden_dim,
                     kernel_size=5, dilation=1,
                     dropout=dropout,
                     drop_path_rate=dp_rates[0]),
        ]
        for i in range(1, num_blocks):
            dilation = min(2 ** (i - 1), 8)
            layers.append(
                TCNBlock(hidden_dim, hidden_dim,
                         kernel_size=3, dilation=dilation,
                         dropout=dropout,
                         drop_path_rate=dp_rates[i])
            )

        self.tcn     = nn.Sequential(*layers)
        self.out_dim = hidden_dim

    def forward(self, x_flat):
        # x_flat: (B, C*J, T)
        tcn_out = self.tcn(x_flat)         # (B, hidden_dim, T)
        return tcn_out.permute(0, 2, 1)    # (B, T, hidden_dim))


# ============================================================
# ST-GCN — Spatial-Temporal Graph Convolutional Network
# ============================================================

def build_adjacency_matrix(num_joints=48):
    """Matrice di adiacenza normalizzata per i 48 joint MediaPipe."""
    A = np.zeros((num_joints, num_joints), dtype=np.float32)

    edges_body          = [(0,1),(1,2),(2,3),(1,4),(4,5)]
    edges_wrist_to_hand = [(3,6),(5,27)]
    finger_base = [
        (0,1),(1,2),(2,3),(3,4),
        (0,5),(5,6),(6,7),(7,8),
        (0,9),(9,10),(10,11),(11,12),
        (0,13),(13,14),(14,15),(15,16),
        (0,17),(17,18),(18,19),(19,20),
    ]
    edges_lh = [(s+6,  e+6)  for s,e in finger_base]
    edges_rh = [(s+27, e+27) for s,e in finger_base]

    for u, v in edges_body + edges_wrist_to_hand + edges_lh + edges_rh:
        if u < num_joints and v < num_joints:
            A[u, v] = 1
            A[v, u] = 1

    # Auto-connessione
    np.fill_diagonal(A, 1)

    # Normalizzazione simmetrica: D^-0.5 * A * D^-0.5
    D = np.diag(A.sum(axis=1) ** -0.5)
    A = D @ A @ D
    return torch.tensor(A, dtype=torch.float32)


class STGCNBlock(nn.Module):
    """
    Blocco ST-GCN: graph conv spaziale + conv temporale.
    Input/output: (B, C, J, T)
    """
    def __init__(self, in_channels, out_channels, A,
                 stride=1, dropout=0.3, residual=True):
        super().__init__()
        self.register_buffer('A', A)

        # Graph conv spaziale: moltiplica per A e proietta canali
        self.gcn = nn.Conv2d(in_channels, out_channels,
                             kernel_size=(1, 1))  # opera su (J, T)

        # Conv temporale: kernel 9x1 (9 frame, 1 joint)
        pad = 4
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels,
                      kernel_size=(1, 9), padding=(0, pad),
                      stride=(1, stride)),
            nn.BatchNorm2d(out_channels),
            nn.Dropout(dropout),
        )

        # Residual
        if not residual:
            self.residual = lambda x: 0
        elif in_channels == out_channels and stride == 1:
            self.residual = nn.Identity()
        else:
            self.residual = nn.Sequential(
                nn.Conv2d(in_channels, out_channels,
                          kernel_size=(1, 1), stride=(1, stride)),
                nn.BatchNorm2d(out_channels),
            )
        self.relu = nn.ReLU()

    def forward(self, x):
        # x: (B, C, J, T)
        res = self.residual(x)
        # Graph conv: A matmul su dimensione J
        x = torch.einsum('bcjt,jk->bckt', x, self.A)
        x = self.gcn(x)   # (B, out_C, J, T)
        x = self.tcn(x)   # (B, out_C, J, T)
        return self.relu(x + res)


class PoseNetworkGCN(nn.Module):
    """
    ST-GCN backbone: sostituisce PoseNetworkTCN.
    Input:  (B, num_channels, num_joints, T)  — NON flattenato
    Output: (B, T, hidden_dim)
    """
    def __init__(self, num_joints=48, in_channels=5,
                 hidden_dim=256, num_blocks=4, dropout=0.3):
        super().__init__()
        A = build_adjacency_matrix(num_joints)

        # Stack di blocchi ST-GCN con canali crescenti
        channels = [in_channels, 64, 128, hidden_dim]
        while len(channels) < num_blocks + 1:
            channels.append(hidden_dim)
        self.blocks = nn.ModuleList()
        for i in range(num_blocks):
            self.blocks.append(
                STGCNBlock(channels[i], channels[i+1], A, dropout=dropout)
            )

        # Pooling spaziale: (B, hidden_dim, J, T) → (B, hidden_dim, T)
        self.spatial_pool = nn.AdaptiveAvgPool2d((1, None))
        self.out_dim = hidden_dim

    def forward(self, x):
        # x: (B, C, J, T) — NON flattenato, diverso da TCN
        for block in self.blocks:
            x = block(x)                           # (B, hidden_dim, J, T)
        x = self.spatial_pool(x).squeeze(2)        # (B, hidden_dim, T)
        return x.permute(0, 2, 1)                  # (B, T, hidden_dim)


class PoseNetworkCTC(nn.Module):
    """
    ST-GCN → TemporalAttention → BiLSTM → CTC.
    """
    def __init__(self, num_classes, num_joints=48, num_channels=3, hidden_dim=256,
                 tcn_blocks=4, lstm_layers=2,
                 dropout=0.3, drop_path_rate=0.1,
                 attn_heads=4):
        super().__init__()

        # ── 1. ST-GCN backbone ───────────────────────────────────────────
        self.gcn_backbone = PoseNetworkGCN(
            num_joints=num_joints,
            in_channels=num_channels,
            hidden_dim=hidden_dim,
            num_blocks=tcn_blocks,
            dropout=dropout,
        )

        # ── 2. Self-Attention temporale ───────────────────────────────────
        self.temporal_attn = TemporalAttention(
            dim=hidden_dim, heads=attn_heads, dropout=dropout * 0.5)

        # ── 3. BiLSTM ────────────────────────────────────────────────────
        self.bilstm = nn.LSTM(
            input_size=hidden_dim, hidden_size=hidden_dim,
            num_layers=lstm_layers, batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0)
        self.norm    = nn.LayerNorm(hidden_dim * 2)
        self.dropout = nn.Dropout(dropout)

        # ── 4. Classificatore principale ──────────────────────────────────
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

        # ── Head ausiliaria CTC con classificatore condiviso ──────────────
        self.aux_proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.Dropout(dropout * 0.5),
        )

    def forward(self, x):
        # x: (B, C, J, T)
        feat = self.gcn_backbone(x)               # (B, T, hidden_dim)

        # ── Attenzione temporale ──────────────────────────────────────────
        feat = self.temporal_attn(feat)           # (B, T, hidden_dim)

        # ── Head ausiliaria CTC ───────────────────────────────────────────
        if self.training:
            aux_feat     = self.aux_proj(feat)    # (B, T, hidden_dim*2)
            aux_log_prob = torch.log_softmax(self.fc(aux_feat), dim=-1)

        # ── BiLSTM → output principale ────────────────────────────────────
        feat, _ = self.bilstm(feat)               # (B, T, hidden_dim*2)
        feat     = self.norm(feat)
        feat     = self.dropout(feat)
        main_log_prob = torch.log_softmax(self.fc(feat), dim=-1)

        if self.training:
            return main_log_prob, aux_log_prob   # run_epoch gestisce già questa tupla
        else:
            return main_log_prob


gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = PoseNetworkCTC(
    num_classes=num_classes,
    num_joints=CONFIG['num_joints'],
    num_channels=CONFIG.get('num_channels', 3),
    hidden_dim=CONFIG['hidden_dim'],
    tcn_blocks=CONFIG['tcn_blocks'],
    lstm_layers=CONFIG['num_layers'],
    dropout=CONFIG['dropout'],
    drop_path_rate=CONFIG['drop_path_rate'],   # ← nuovo
    attn_heads=CONFIG['attn_heads'],           # ← nuovo
).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parametri totali:    {total_params:,}')
print(f'Parametri trainable: {trainable_params:,}')
print(f'✓ PoseNetworkCTC istanziato su {DEVICE}')

✓ run_epoch definita e pronta all'uso
✓ run_epoch_clean() definita e pronta all'uso
Parametri totali:    6,468,862
Parametri trainable: 6,468,862
✓ PoseNetworkCTC istanziato su cuda


In [ ]:
# ============================================================
# STEP 8b — Self-Supervised Pre-training (Masked Joint)
# ============================================================

class MaskedJointPretrainer(nn.Module):
    """
    Wrapper per pre-training: usa il backbone GCN/TCN per
    ricostruire i joint mascherati. Testa di regressione
    separata, scartata dopo il pre-training.
    """
    def __init__(self, backbone, hidden_dim, num_joints, num_channels):
        super().__init__()
        self.backbone   = backbone
        self.recon_head = nn.Linear(hidden_dim, num_joints * num_channels)
        self.num_joints   = num_joints
        self.num_channels = num_channels

    def forward(self, x, mask):
        # x:    (B, C, J, T)
        # mask: (B, J) boolean — joint da mascherare
        x_masked = x.masked_fill(mask[:, None, :, None], 0.0)
        feat = self.backbone(x_masked)       # (B, T, hidden_dim)
        feat = feat.mean(dim=1)              # pooling temporale → (B, hidden_dim)
        recon = self.recon_head(feat)        # (B, J*C)
        recon = recon.view(-1, self.num_joints, self.num_channels)
        return recon


def pretrain_one_epoch(model, dl, optimizer, device, mask_ratio=0.3):
    model.train()
    total_loss = 0.0
    for tssies, *_ in tqdm(dl, desc='Pretrain', leave=False):
        tssies = tssies.to(device)   # (B, C, J, T)
        B, C, J, T = tssies.shape

        # Crea maschera casuale su J joint
        mask = torch.rand(B, J, device=device) < mask_ratio

        optimizer.zero_grad(set_to_none=True)
        recon = model(tssies, mask)   # (B, J, C)

        # Target: valori originali dei joint mascherati (media temporale)
        target = tssies.permute(0, 2, 1, 3).mean(dim=-1)  # (B, J, C)

        # Loss solo sui joint mascherati
        loss = F.mse_loss(recon[mask], target[mask])
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(dl)


# Avvio pre-training (5–10 epoche bastano)
PRETRAIN_EPOCHS = 7
pretrain_model = MaskedJointPretrainer(
    backbone   = model.gcn_backbone,
    hidden_dim = CONFIG['hidden_dim'],
    num_joints = CONFIG['num_joints'],
    num_channels = CONFIG.get('num_channels', 5),
).to(DEVICE)

opt_pretrain = torch.optim.AdamW(pretrain_model.parameters(), lr=3e-4)

print('=== Self-Supervised Pre-training ===')
for ep in range(PRETRAIN_EPOCHS):
    loss = pretrain_one_epoch(pretrain_model, train_dl, opt_pretrain, DEVICE)
    print(f'  Pretrain ep {ep+1}/{PRETRAIN_EPOCHS} | MSE loss {loss:.4f}')

print('✓ Pre-training completato — backbone aggiornato')

In [20]:
# ── Debug pre-training: un batch campione per train/dev ───────────────────
def summarize_label_distribution(dataset, i2g, top_k=20):
    counter = Counter()
    for sample in dataset.samples:
        counter.update(sample['labels'])
    print(f'Classi osservate nel dataset: {len(counter)}')
    print(f'Top-{top_k} label più frequenti:')
    for idx, count in counter.most_common(top_k):
        print(f'  {i2g.get(idx, idx):<20} {count}')
    return counter

def print_epoch_summary(epoch, metrics, loss, wer, training=True):
    split = 'TRAIN' if training else 'VAL'
    print(
        f'[{split}] Ep {epoch:3d} | '
        f'Loss {loss:.4f} | WER {wer*100:.2f}% | '
        f'Classi corrette: {metrics["correct_classes"]}/{metrics["total_classes"]} '
        f'({metrics["class_accuracy"]:.1f}%) | '
        f'Pred token: {metrics["total_pred"]}'
    )
def preview_one_batch(model, dataloader, split_name, device, config, log_prior, i2g):
    model.eval()
    with torch.no_grad():
        for batch_idx, (tssies, targets, input_lengths, target_lengths) in enumerate(dataloader):
            tssies = tssies.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16,
                                enabled=config['use_amp'] and device.type == 'cuda'):
                log_probs = model(tssies)
                lp_float = log_probs.permute(1, 0, 2).float()

            if split_name == 'train':
                decoded = greedy_decode_with_prior(lp_float.cpu(), beta=0.0)
            else:
                decoded = greedy_decode_with_prior(lp_float.cpu(), beta=config['prior_beta'], log_prior=log_prior.cpu())

            refs = []
            offset = 0
            for tlen in target_lengths.tolist():
                refs.append(targets[offset:offset + tlen].tolist())
                offset += tlen

            frame_argmax = lp_float.argmax(dim=-1)
            frame_counter = Counter(frame_argmax.reshape(-1).tolist())
            pred_counter = Counter()
            ref_counter = Counter()
            for seq in decoded:
                pred_counter.update(seq)
            for seq in refs:
                ref_counter.update(seq)

            print(f'\n=== Preview {split_name.upper()} batch {batch_idx + 1} ===')
            print(f'Loss shape logits: {tuple(lp_float.shape)} | batch size: {len(refs)}')
            print(f'Frame argmax top-10: {format_counter(frame_counter, i2g, 10)}')
            print(f'Ref top-10:          {format_counter(ref_counter, i2g, 10)}')
            print(f'Pred top-10:         {format_counter(pred_counter, i2g, 10)}')

            preview_n = min(config['debug_preview_samples'], len(refs))
            for sample_idx in range(preview_n):
                print(f'  sample {sample_idx + 1:02d}')
                print(f'    REF: {ids_to_gloss_text(refs[sample_idx], i2g)}')
                print(f'    HYP: {ids_to_gloss_text(decoded[sample_idx], i2g)}')

            break


train_label_counter = summarize_label_distribution(train_ds, i2g, top_k=20)
print('\nPreview batch train:')
preview_one_batch(model, train_dl, 'train', DEVICE, CONFIG, LOG_PRIOR, i2g)
print('\nPreview batch dev:')
preview_one_batch(model, dev_dl, 'dev', DEVICE, CONFIG, LOG_PRIOR, i2g)


Classi osservate nel dataset: 703
Top-20 label più frequenti:
  REGEN                676
  REGION               646
  IX                   515
  ANKOMMEN             449
  NORD                 414
  MORGEN               407
  ACH                  360
  SONNE                357
  WOLKE                338
  GRAD                 324
  NACHT                276
  SUED                 275
  SCHNEE               262
  KOENNEN              256
  MEHR                 252
  BISSCHEN             250
  HEUTE                237
  UNWETTER             236
  BIS                  207
  GEWITTER             203

Preview batch train:

=== Preview TRAIN batch 1 ===
Loss shape logits: (190, 16, 1022) | batch size: 16
Frame argmax top-10: SEIT:1541, BESTE:451, STRENG:312, LAUFEN:256, F+E+R+Z:110, ORT:66, A+Z:66, ITALIEN:59, OBER:36, WESER:35
Ref top-10:          WOLKE:6, REGEN:6, IX:5, ACH:5, REGION:4, GEWITTER:4, ANKOMMEN:4, OST:3, KOENNEN:3, MORGEN:3
Pred top-10:         SEIT:54, LAUFEN:30, STRENG:25, OR

In [ ]:
# ============================================================
# FUNZIONI DI UTILITÀ — richieste da run_epoch
# ============================================================

def collapse_ctc(token_ids):
    """Rimuove blank (0) e duplicati consecutivi."""
    result, prev = [], None
    for idx in token_ids:
        v = idx.item() if hasattr(idx, 'item') else int(idx)
        if v != 0 and v != prev:
            result.append(v)
        prev = v
    return result

def print_epoch_summary(epoch, metrics, loss, wer, training=True):
    split = 'TRAIN' if training else 'VAL'
    
    # Estraiamo i dettagli se ci sono
    details = metrics.get('wer_details', {'S': 0, 'D': 0, 'I': 0, 'N': 1})
    N = details['N']
    s_pct = (details['S'] / N) * 100
    d_pct = (details['D'] / N) * 100
    i_pct = (details['I'] / N) * 100
    
    print(
        f'[{split}] Ep {epoch:3d} | Loss {loss:.4f} | WER {wer*100:.2f}% \n'
        f'    ↳ Breakdown: Sost={s_pct:.1f}% | Canc={d_pct:.1f}% | Ins={i_pct:.1f}% \n'
        f'    ↳ Classi corrette: {metrics["correct_classes"]}/{metrics["total_classes"]} '
        f'({metrics["class_accuracy"]:.1f}%) | Pred token: {metrics["total_pred"]}'
    )
def greedy_decode_with_prior(log_probs_TBC, beta=0.0, log_prior=None):
    """Greedy CTC decode con prior scaling opzionale."""
    T, B, C = log_probs_TBC.shape
    results = []
    for b in range(B):
        lp = log_probs_TBC[:, b, :].clone()
        if beta > 0.0 and log_prior is not None:
            lp = lp - beta * log_prior.unsqueeze(0)
        best_ids = lp.argmax(dim=-1).tolist()
        results.append(collapse_ctc(best_ids))
    return results





def compute_epoch_metrics(all_refs, all_hyps, all_pred_counter,
                           all_ref_counter, i2g, top_k=8):
    """Calcola metriche aggregate per l'epoch summary."""
    total_ref_tokens  = sum(all_ref_counter.values())
    total_pred_tokens = sum(all_pred_counter.values())

    ref_dist  = {i2g.get(k, k): round(v / max(total_ref_tokens,  1) * 100, 2)
                 for k, v in all_ref_counter.most_common(top_k)}
    pred_dist = {i2g.get(k, k): round(v / max(total_pred_tokens, 1) * 100, 2)
                 for k, v in all_pred_counter.most_common(top_k)}

    # Quante classi uniche il modello ha predetto correttamente (almeno una volta)
    ref_set  = set(all_ref_counter.keys())
    pred_set = set(all_pred_counter.keys())
    correct  = len(ref_set & pred_set)

    return {
        'correct_classes': correct,
        'total_classes':   len(ref_set),
        'class_accuracy':  correct / max(len(ref_set), 1) * 100,
        'ref_dist':        ref_dist,
        'pred_dist':       pred_dist,
        'total_pred':      total_pred_tokens,
    }


def ids_to_gloss_text(ids, i2g):
    return ' '.join(i2g.get(i, f'[{i}]') for i in ids)


def format_counter(counter, i2g, top_k=10):
    return ', '.join(
        f'{i2g.get(k, k)}:{v}'
        for k, v in counter.most_common(top_k)
    )


# ── Two-phase helpers per TSSICTCModel ──────────────────────────────────

def freeze_backbone(model):
    for module in [model.gcn_backbone, model.temporal_attn]:
        for p in module.parameters():
            p.requires_grad = False
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'[Phase 1] Backbone congelato — trainable: {n:,} params')


def unfreeze_backbone(model):
    for p in model.parameters():
        p.requires_grad = True
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'[Phase 2] Tutto sbloccato — trainable: {n:,} params')


def make_optimizer_phase1(model, lr, weight_decay):
    """Fase 1: solo BiLSTM + norm + FC."""
    params = [p for p in model.parameters() if p.requires_grad]
    return torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)


def make_optimizer_phase2(model, lr_backbone, lr_head, weight_decay):
    """Fase 2: backbone lento, testa veloce."""
    return torch.optim.AdamW([
    {'params': model.gcn_backbone.parameters(),  'lr': lr_backbone},
    {'params': model.temporal_attn.parameters(), 'lr': lr_backbone},
    {'params': model.bilstm.parameters(),        'lr': lr_head},
    {'params': model.norm.parameters(),          'lr': lr_head},
    {'params': model.fc.parameters(),            'lr': lr_head},
    {'params': model.aux_proj.parameters(),      'lr': lr_head},
], weight_decay=weight_decay)
def make_scheduler(optimizer, num_epochs, steps_per_epoch):
    from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
    warmup = LinearLR(optimizer, start_factor=0.1,
                      total_iters=5 * steps_per_epoch)
    cosine = CosineAnnealingLR(optimizer,
                               T_max=num_epochs * steps_per_epoch,
                               eta_min=1e-6)
    return SequentialLR(optimizer, [warmup, cosine],
                        milestones=[5 * steps_per_epoch])


class CTCLossWithSmoothing(nn.Module):
    """Wrapper per compatibilità — usa CTCLossWithEntropy internamente."""
    def __init__(self, blank=0, smoothing=0.05):
        super().__init__()
        self.ctc = CTCLossWithEntropy(blank=blank, entropy_weight=smoothing)

    def forward(self, log_probs_TBC, targets, input_lengths, target_lengths):
        return self.ctc(log_probs_TBC, targets, input_lengths, target_lengths)


print('✓ Funzioni di utilità definite')

✓ Funzioni di utilità definite


## Step 9 — Avvio Training (Two-Phase)

In [ ]:
# ============================================================================
# VERO TWO-PHASE TRAINING LOOP
# ============================================================================
def freeze_backbone(model):
    # Congela backbone e attenzione temporale
    for module in [model.gcn_backbone, model.temporal_attn]:
        for p in module.parameters():
            p.requires_grad = False
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Backbone congelato — parametri trainable: {n:,}')


def unfreeze_backbone(model):
    for p in model.parameters():
        p.requires_grad = True
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Backbone sbloccato — parametri trainable: {n:,}')
# ── Class weights per classi rare ─────────────────────────────────────────
all_labels_flat = [lbl for row in train_df_merged['orth'] for lbl in row]
label_counts    = Counter(all_labels_flat)
max_count       = max(label_counts.values())

class_weights = torch.ones(num_classes, device=DEVICE)
for gloss, idx in g2i.items():
    if gloss in label_counts:
        # radice quadrata per smorzare: rari pesano di più ma non troppo
        class_weights[idx] = min((max_count / label_counts[gloss]) ** 0.5, 5.0)

print(f'✓ Class weights calcolati | min={class_weights.min():.2f} max={class_weights.max():.2f}')

# Passa i pesi al criterion

criterion = CTCLossWithSmoothing(blank=0, smoothing=CONFIG['ctc_smoothing'])
scaler    = GradScaler(enabled=CONFIG['use_amp'] and DEVICE.type == 'cuda')

best_wer  = float('inf')
patience  = 0
history   = {'train_loss': [], 'dev_loss': [], 'dev_wer': [], 'phase': []}
checkpoint_paths = []
global_epoch = 0

print('=' * 60)
print(f'INIZIO FASE 1: {CONFIG["phase1_epochs"]} epoch, BACKBONE CONGELATO')
print('=' * 60)

# FASE 1: Congela il TCN, allena solo la testa
freeze_backbone(model)
optimizer_p1 = make_optimizer_phase1(model, lr=CONFIG['phase1_lr'], weight_decay=CONFIG['weight_decay'])
scheduler_p1 = make_scheduler(optimizer_p1, CONFIG['phase1_epochs'], len(train_dl))

for epoch in range(CONFIG['phase1_epochs']):
    global_epoch += 1
    
    # 1. Unpack all 7 variables for TRAIN
    (train_loss, train_wer, train_metrics,
     train_wm, train_pc, train_never, train_low,
     train_wm_unk, train_pc_unk, train_never_unk, train_low_unk) = run_epoch(
        model, train_dl, criterion, optimizer_p1, scaler, scheduler_p1,
        training=True, device=DEVICE, config=CONFIG, log_prior=LOG_PRIOR, i2g=i2g)
        
    # 2. Unpack all 7 variables for DEV
    (dev_loss, dev_wer, dev_metrics,
     dev_wm, dev_pc, dev_never, dev_low,
     dev_wm_unk, dev_pc_unk, dev_never_unk, dev_low_unk) = run_epoch(
        model, dev_dl, criterion, optimizer_p1, scaler, scheduler_p1,
        training=False, device=DEVICE, config=CONFIG, log_prior=LOG_PRIOR, i2g=i2g)

    print_epoch_summary(global_epoch, train_metrics, train_loss, train_wer, training=True)
    print_epoch_summary(global_epoch, dev_metrics, dev_loss, dev_wer, training=False)
    
    # 3. Word metrics (con e senza <unk> come classe valida)
    print_word_metrics(dev_wm, dev_pc, dev_never, dev_low,
                       summary_unk=dev_wm_unk, per_class_unk=dev_pc_unk,
                       never_predicted_unk=dev_never_unk, low_recall_unk=dev_low_unk,
                       epoch=global_epoch, split='VAL')

    history['train_loss'].append(train_loss)
    history['dev_loss'].append(dev_loss)
    history['dev_wer'].append(dev_wer)
    history['phase'].append(1)

    print(f'[FASE 1] Ep {epoch+1:2d}/{CONFIG["phase1_epochs"]} (Global {global_epoch}) | Loss {train_loss:.4f}/{dev_loss:.4f} | WER {train_wer*100:.1f}%/{dev_wer*100:.1f}%')

    if dev_wer < best_wer:
        best_wer = dev_wer
        torch.save(model.state_dict(), os.path.join(RESULTS_DIR, 'best_model.pth'))
        print(f'  → Nuovo best WER: {best_wer*100:.2f}%')


print('\\n' + '=' * 60)
phase2_epochs = CONFIG['num_epochs'] - CONFIG['phase1_epochs']
print(f'INIZIO FASE 2: {phase2_epochs} epoch, TUTTO SBLOCCATO (LR differenziale)')
print('=' * 60)

# FASE 2: Sblocca il TCN, abbassa il LR
unfreeze_backbone(model)
optimizer_p2 = make_optimizer_phase2(
    model,
    lr_backbone=CONFIG['phase2_lr_backbone'],
    lr_head=CONFIG['phase2_lr_head'],
    weight_decay=CONFIG['weight_decay']
)
scheduler_p2 = make_scheduler(optimizer_p2, phase2_epochs, len(train_dl))

for epoch in range(phase2_epochs):
    global_epoch += 1
       # 1. Unpack all 7 variables for TRAIN
    (train_loss, train_wer, train_metrics,
     train_wm, train_pc, train_never, train_low,
     train_wm_unk, train_pc_unk, train_never_unk, train_low_unk) = run_epoch(
        model, train_dl, criterion, optimizer_p2, scaler, scheduler_p2,
        training=True, device=DEVICE, config=CONFIG, log_prior=LOG_PRIOR, i2g=i2g)
        
    # 2. Unpack all 7 variables for DEV
    (dev_loss, dev_wer, dev_metrics,
     dev_wm, dev_pc, dev_never, dev_low,
     dev_wm_unk, dev_pc_unk, dev_never_unk, dev_low_unk) = run_epoch(
        model, dev_dl, criterion, optimizer_p2, scaler, scheduler_p2,
        training=False, device=DEVICE, config=CONFIG, log_prior=LOG_PRIOR, i2g=i2g)

    print_epoch_summary(global_epoch, train_metrics, train_loss, train_wer, training=True)
    print_epoch_summary(global_epoch, dev_metrics, dev_loss, dev_wer, training=False)
    
    # Word metrics (con e senza <unk> come classe valida)
    print_word_metrics(dev_wm, dev_pc, dev_never, dev_low,
                       summary_unk=dev_wm_unk, per_class_unk=dev_pc_unk,
                       never_predicted_unk=dev_never_unk, low_recall_unk=dev_low_unk,
                       epoch=global_epoch, split='VAL')

    history['train_loss'].append(train_loss)
    history['dev_loss'].append(dev_loss)
    history['dev_wer'].append(dev_wer)
    history['phase'].append(2)

    print(f'[FASE 2] Ep {epoch+1:2d}/{phase2_epochs} (Global {global_epoch}) | Loss {train_loss:.4f}/{dev_loss:.4f} | WER {train_wer*100:.1f}%/{dev_wer*100:.1f}%')
   
    # Checkpoint per ensemble
    ckpt_path = os.path.join(RESULTS_DIR, f'checkpoint_ep{global_epoch:03d}.pth')
    torch.save(model.state_dict(), ckpt_path)
    checkpoint_paths.append((ckpt_path, dev_wer))

    if len(checkpoint_paths) > CONFIG['keep_last_n_checkpoints']:
        old_path, _ = checkpoint_paths.pop(0)
        if os.path.exists(old_path):
            os.remove(old_path)

    if dev_wer < best_wer:
        best_wer = dev_wer
        patience  = 0
        torch.save(model.state_dict(), os.path.join(RESULTS_DIR, 'best_model.pth'))
        print(f'  → Nuovo best WER: {best_wer*100:.2f}%')
    else:
        patience += 1
        if patience >= CONFIG['early_stopping_patience']:
            print(f'Early stopping a epoch {global_epoch}')
            break

    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


print('\\n' + '=' * 60)
PHASE3_EPOCHS = 20
phase3_lr     = CONFIG['phase2_lr_backbone'] * 0.1

optimizer_p3 = torch.optim.AdamW([
    {'params': model.gcn_backbone.parameters(),  'lr': phase3_lr},
    {'params': model.temporal_attn.parameters(), 'lr': phase3_lr},
    {'params': model.bilstm.parameters(),        'lr': phase3_lr * 3},
    {'params': model.norm.parameters(),          'lr': phase3_lr * 3},
    {'params': model.fc.parameters(),            'lr': phase3_lr * 3},
    {'params': model.aux_proj.parameters(),      'lr': phase3_lr * 3},
], weight_decay=CONFIG['weight_decay'] * 2)

scheduler_p3 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_p3, T_max=PHASE3_EPOCHS, eta_min=phase3_lr * 0.01)

print('=== FASE 3: Fine-tuning a LR basso ===')
for epoch in range(PHASE3_EPOCHS):
    global_epoch += 1

    current_smoothing = 0.05 + 0.10 * (epoch / PHASE3_EPOCHS)
    criterion_p3 = CTCLossWithSmoothing(blank=0, smoothing=current_smoothing)

    (train_loss, train_wer, *_) = run_epoch(
        model, train_dl, criterion_p3, optimizer_p3, scaler, scheduler_p3,
        training=True, device=DEVICE, config=CONFIG, log_prior=LOG_PRIOR, i2g=i2g)

    (dev_loss, dev_wer, *_) = run_epoch(
        model, dev_dl, criterion_p3, optimizer_p3, scaler, scheduler_p3,
        training=False, device=DEVICE, config=CONFIG, log_prior=LOG_PRIOR, i2g=i2g)

    scheduler_p3.step()

    history['train_loss'].append(train_loss)
    history['dev_loss'].append(dev_loss)
    history['dev_wer'].append(dev_wer)
    history['phase'].append(3)

    print(f'[FASE 3] Ep {epoch+1}/{PHASE3_EPOCHS} | '
          f'Loss {train_loss:.4f}/{dev_loss:.4f} | '
          f'WER {train_wer*100:.1f}%/{dev_wer*100:.1f}% | '
          f'smoothing={current_smoothing:.3f}')

    if dev_wer < best_wer:
        best_wer = dev_wer
        torch.save(model.state_dict(),
                   os.path.join(RESULTS_DIR, 'best_model_p3.pth'))
        print(f'  → Nuovo best WER: {best_wer*100:.2f}%')

    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f'\\n✓ Training completato. Best WER: {best_wer*100:.2f}%')

✓ Class weights calcolati | min=1.00 max=5.00
INIZIO FASE 1: 15 epoch, BACKBONE CONGELATO
  Backbone congelato — parametri trainable: 4,864,510


Train:   0%|          | 0/141 [00:00<?, ?it/s]

[TRAIN] Ep   1 | Loss 31.7570 | WER 483.16% 
    ↳ Breakdown: Sost=22.0% | Canc=76.9% | Ins=384.2% 
    ↳ Classi corrette: 448/703 (63.7%) | Pred token: 72651
[VAL] Ep   1 | Loss 5.6655 | WER 100.00% 
    ↳ Breakdown: Sost=0.0% | Canc=100.0% | Ins=0.0% 
    ↳ Classi corrette: 0/365 (0.0%) | Pred token: 0

─────────────────────────────────────────────────────────────────
[WORD METRICS VAL Ep 1]  (escluso <unk>)
  Macro-F1:        0.00%
  Macro-Precision: 0.00%
  Macro-Recall:    0.00%
  Classi totali:   365
  Mai predette:    152  (support ≥ 5)
  Low recall<20%:  0  (support ≥ 5)

  Top-15 classi per support:
  Gloss                    Sup   Prec    Rec     F1
  ──────────────────────────────────────────────────
  REGEN                    128   0.0%   0.0%   0.0%
  REGION                   124   0.0%   0.0%   0.0%
  MORGEN                   109   0.0%   0.0%   0.0%
  IX                        97   0.0%   0.0%   0.0%
  GRAD                      92   0.0%   0.0%   0.0%
  NORD             

[TRAIN] Ep   2 | Loss 5.5359 | WER 100.00% 
    ↳ Breakdown: Sost=0.0% | Canc=100.0% | Ins=0.0% 
    ↳ Classi corrette: 0/703 (0.0%) | Pred token: 0
[VAL] Ep   2 | Loss 5.3928 | WER 100.00% 
    ↳ Breakdown: Sost=0.0% | Canc=100.0% | Ins=0.0% 
    ↳ Classi corrette: 0/365 (0.0%) | Pred token: 0

─────────────────────────────────────────────────────────────────
[WORD METRICS VAL Ep 2]  (escluso <unk>)
  Macro-F1:        0.00%
  Macro-Precision: 0.00%
  Macro-Recall:    0.00%
  Classi totali:   365
  Mai predette:    152  (support ≥ 5)
  Low recall<20%:  0  (support ≥ 5)

  Top-15 classi per support:
  Gloss                    Sup   Prec    Rec     F1
  ──────────────────────────────────────────────────
  REGEN                    128   0.0%   0.0%   0.0%
  REGION                   124   0.0%   0.0%   0.0%
  MORGEN                   109   0.0%   0.0%   0.0%
  IX                        97   0.0%   0.0%   0.0%
  GRAD                      92   0.0%   0.0%   0.0%
  NORD                      8

[TRAIN] Ep   3 | Loss 5.3953 | WER 100.00% 
    ↳ Breakdown: Sost=0.0% | Canc=100.0% | Ins=0.0% 
    ↳ Classi corrette: 0/703 (0.0%) | Pred token: 0
[VAL] Ep   3 | Loss 5.2610 | WER 100.00% 
    ↳ Breakdown: Sost=0.0% | Canc=100.0% | Ins=0.0% 
    ↳ Classi corrette: 0/365 (0.0%) | Pred token: 0

─────────────────────────────────────────────────────────────────
[WORD METRICS VAL Ep 3]  (escluso <unk>)
  Macro-F1:        0.00%
  Macro-Precision: 0.00%
  Macro-Recall:    0.00%
  Classi totali:   365
  Mai predette:    152  (support ≥ 5)
  Low recall<20%:  0  (support ≥ 5)

  Top-15 classi per support:
  Gloss                    Sup   Prec    Rec     F1
  ──────────────────────────────────────────────────
  REGEN                    128   0.0%   0.0%   0.0%
  REGION                   124   0.0%   0.0%   0.0%
  MORGEN                   109   0.0%   0.0%   0.0%
  IX                        97   0.0%   0.0%   0.0%
  GRAD                      92   0.0%   0.0%   0.0%
  NORD                      8

...

[TRAIN] Ep  87 | Loss 1.7538 | WER 27.38% 
    ↳ Breakdown: Sost=9.9% | Canc=15.0% | Ins=2.5% 
    ↳ Classi corrette: 672/703 (95.6%) | Pred token: 15599
[VAL] Ep  87 | Loss 4.8793 | WER 76.00% 
    ↳ Breakdown: Sost=23.4% | Canc=52.3% | Ins=0.3% 
    ↳ Classi corrette: 190/365 (52.1%) | Pred token: 1791

─────────────────────────────────────────────────────────────────
[WORD METRICS VAL Ep 87]  (escluso <unk>)
  Macro-F1:        31.50%
  Macro-Precision: 46.50%
  Macro-Recall:    25.36%
  Classi totali:   398
  Mai predette:    19  (support ≥ 5)
  Low recall<20%:  37  (support ≥ 5)

  Top-15 classi per support:
  Gloss                    Sup   Prec    Rec     F1
  ──────────────────────────────────────────────────
  REGEN                    128  59.6%  48.4%  53.4%
  REGION                   124  52.5%  25.8%  34.6%
  MORGEN                   109  87.9%  46.8%  61.1%
  IX                        97  45.0%   9.3%  15.4%
  GRAD                      92  80.4%  44.6%  57.3%
  NORD         

[TRAIN] Ep  88 | Loss 1.9930 | WER 37.27% 
    ↳ Breakdown: Sost=16.3% | Canc=16.9% | Ins=4.1% 
    ↳ Classi corrette: 664/702 (94.6%) | Pred token: 15555
[VAL] Ep  88 | Loss 4.7983 | WER 74.48% 
    ↳ Breakdown: Sost=24.4% | Canc=49.5% | Ins=0.6% 
    ↳ Classi corrette: 185/365 (50.7%) | Pred token: 1908

─────────────────────────────────────────────────────────────────
[WORD METRICS VAL Ep 88]  (escluso <unk>)
  Macro-F1:        32.91%
  Macro-Precision: 47.37%
  Macro-Recall:    27.24%
  Classi totali:   406
  Mai predette:    18  (support ≥ 5)
  Low recall<20%:  43  (support ≥ 5)

  Top-15 classi per support:
  Gloss                    Sup   Prec    Rec     F1
  ──────────────────────────────────────────────────
  REGEN                    128  57.8%  40.6%  47.7%
  REGION                   124  57.9%  35.5%  44.0%
  MORGEN                   109  74.1%  55.0%  63.2%
  IX                        97  38.1%   8.2%  13.6%
  GRAD                      92  84.4%  41.3%  55.5%
  NORD        

[TRAIN] Ep  89 | Loss 1.7622 | WER 30.83% 
    ↳ Breakdown: Sost=13.3% | Canc=13.8% | Ins=3.7% 
    ↳ Classi corrette: 676/703 (96.2%) | Pred token: 16035
[VAL] Ep  89 | Loss 4.7813 | WER 74.69% 
    ↳ Breakdown: Sost=25.8% | Canc=48.2% | Ins=0.7% 
    ↳ Classi corrette: 185/365 (50.7%) | Pred token: 1959

─────────────────────────────────────────────────────────────────
[WORD METRICS VAL Ep 89]  (escluso <unk>)
  Macro-F1:        33.16%
  Macro-Precision: 46.94%
  Macro-Recall:    27.34%
  Classi totali:   405
  Mai predette:    16  (support ≥ 5)
  Low recall<20%:  41  (support ≥ 5)

  Top-15 classi per support:
  Gloss                    Sup   Prec    Rec     F1
  ──────────────────────────────────────────────────
  REGEN                    128  60.6%  44.5%  51.3%
  REGION                   124  52.6%  33.1%  40.6%
  MORGEN                   109  73.5%  56.0%  63.5%
  IX                        97  42.3%  11.3%  17.9%
  GRAD                      92  78.7%  40.2%  53.2%
  NORD        

[TRAIN] Ep  90 | Loss 1.8421 | WER 30.77% 
    ↳ Breakdown: Sost=12.9% | Canc=14.9% | Ins=3.0% 
    ↳ Classi corrette: 665/703 (94.6%) | Pred token: 15717
[VAL] Ep  90 | Loss 4.8240 | WER 74.13% 
    ↳ Breakdown: Sost=24.9% | Canc=48.9% | Ins=0.3% 
    ↳ Classi corrette: 190/365 (52.1%) | Pred token: 1921

─────────────────────────────────────────────────────────────────
[WORD METRICS VAL Ep 90]  (escluso <unk>)
  Macro-F1:        33.46%
  Macro-Precision: 47.31%
  Macro-Recall:    27.37%
  Classi totali:   405
  Mai predette:    14  (support ≥ 5)
  Low recall<20%:  37  (support ≥ 5)

  Top-15 classi per support:
  Gloss                    Sup   Prec    Rec     F1
  ──────────────────────────────────────────────────
  REGEN                    128  58.8%  46.9%  52.2%
  REGION                   124  57.8%  33.1%  42.0%
  MORGEN                   109  74.1%  57.8%  65.0%
  IX                        97  38.7%  12.4%  18.8%
  GRAD                      92  77.3%  37.0%  50.0%
  NORD        

KeyboardInterrupt: 

## Step 10 — Model Ensemble (Deep Sign §6.5)

In [ ]:
def predict_with_tta(model, tssi_batch, n_aug=4):
    """Inference con media su N augmentazioni — riduce varianza senza ritraining."""
    model.eval()
    log_probs_list = []
    with torch.no_grad():
        # Predizione originale (no augment)
        lp = model(tssi_batch)
        if isinstance(lp, tuple):
            lp = lp[0]
        log_probs_list.append(lp)
        # N versioni augmentate
        for _ in range(n_aug - 1):
            aug = tssi_batch.cpu().numpy().copy()
            for i in range(aug.shape[0]):
                aug[i] = augment_tssi_fixed(aug[i], augment=True)
            aug_t = torch.from_numpy(aug).to(tssi_batch.device)
            lp_aug = model(aug_t)
            if isinstance(lp_aug, tuple):
                lp_aug = lp_aug[0]
            log_probs_list.append(lp_aug)
    return torch.stack(log_probs_list).mean(dim=0)



def ensemble_decode(model_paths, dl, device, config, log_prior, num_classes):
    """
    Carica N checkpoint e media le log-prob (product of probabilities).
    Equivale al log-linear combination del Deep Sign §6.5.

    Il paper usa δ1=0.87, δ2=0.13 per i due modelli migliori;
    qui usiamo media uniforme (δ=1/N) come baseline.
    Per risultati migliori puoi ottimizzare i δ sul dev set.
    """
    # Carica tutti i modelli
    ensemble = []
    for path in model_paths:
        m = PoseNetworkCTC(
            num_classes=num_classes,
            num_joints=config['num_joints'],
            num_channels=config.get('num_channels', 3),
            hidden_dim=config['hidden_dim'],
            tcn_blocks=config['tcn_blocks'],
            lstm_layers=config['num_layers'],
            dropout=0.0,
        ).to(device)
        m.load_state_dict(torch.load(path, map_location=device))
        m.eval()
        ensemble.append(m)
    print(f'Ensemble: {len(ensemble)} modelli caricati')

    beta     = config['prior_beta']
    lp_dev   = log_prior.to(device)
    all_refs, all_hyps = [], []

    with torch.no_grad():
        for tssies, targets, input_lengths, target_lengths in tqdm(dl, desc='Ensemble'):
            tssies = tssies.to(device)

            # Media delle log-probabilità
            avg_lp = None
            for m in ensemble:
                lp = m(tssies).permute(1, 0, 2).float()  # (T', B, C)
                avg_lp = lp if avg_lp is None else avg_lp + lp
            avg_lp = avg_lp / len(ensemble)

            decoded = beam_search_ctc_optimized(
                avg_lp.cpu(), beam_width=config['beam_width'],
                beta=beta, log_prior=log_prior.cpu())

            refs, offset = [], 0
            for tlen in target_lengths.tolist():
                refs.append(targets[offset:offset+tlen].tolist())
                offset += tlen
            all_refs.extend(refs)
            all_hyps.extend(decoded)

            del tssies, avg_lp
            gc.collect()

    return compute_wer(all_refs, all_hyps)


# Scegli i migliori N checkpoint per l'ensemble
N_ensemble = CONFIG['ensemble_n']
best_ckpts = sorted(checkpoint_paths, key=lambda x: x[1])[:N_ensemble]
ensemble_paths = [p for p, _ in best_ckpts]
print(f'Checkpoint selezionati per ensemble (migliori {N_ensemble} per WER dev):')
for p, w in best_ckpts:
    print(f'  {os.path.basename(p)}  WER dev={w*100:.2f}%')

if len(ensemble_paths) >= 2:
    ensemble_dev_wer = ensemble_decode(
        ensemble_paths, dev_dl, DEVICE, CONFIG, LOG_PRIOR, num_classes)
    print(f'\n📊 Ensemble WER (dev):  {ensemble_dev_wer*100:.2f}%')
    print(f'📊 Best single WER (dev): {best_wer*100:.2f}%')
    print(f'📊 Guadagno ensemble:   {(best_wer - ensemble_dev_wer)*100:.2f}% abs')
else:
    print('⚠ Non abbastanza checkpoint per ensemble (servono ≥ 2)')

## Step 11 — Valutazione sul Test Set

In [ ]:
print('=' * 60 + '\nTEST SET EVALUATION\n' + '=' * 60)

# ── Modello singolo (best checkpoint) ────────────────────────────────────
best_path = os.path.join(RESULTS_DIR, 'best_model.pth')
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
model.eval()

test_wer_ensemble = None   # inizializzato qui, assegnato solo se ensemble disponibile

best_path = os.path.join(RESULTS_DIR, 'best_model.pth')
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
model.eval()

(test_loss, test_wer_single, test_metrics,
 test_wm, test_pc, test_never, test_low,
 test_wm_unk, test_pc_unk, test_never_unk, test_low_unk) = run_epoch(
    model, test_dl, criterion, None, scaler, None,
    training=False, device=DEVICE, config=CONFIG, log_prior=LOG_PRIOR, i2g=i2g)

# Word metrics sul test set
print_word_metrics(test_wm, test_pc, test_never, test_low,
                   summary_unk=test_wm_unk, per_class_unk=test_pc_unk,
                   never_predicted_unk=test_never_unk, low_recall_unk=test_low_unk,
                   split='TEST')

print(f'📊 Test WER (singolo): {test_wer_single*100:.2f}%')



# ── Ensemble ─────────────────────────────────────────────────────────────
if len(ensemble_paths) >= 2:
    test_wer_ensemble = ensemble_decode(
        ensemble_paths, test_dl, DEVICE, CONFIG, LOG_PRIOR, num_classes)
    print(f'📊 Test WER (ensemble {len(ensemble_paths)} modelli): {test_wer_ensemble*100:.2f}%')
    

# ── Salva risultati ───────────────────────────────────────────────────────
results = {
    'config':              CONFIG,
    'raw_gloss_count':     int(raw_gloss_count),
    'merged_gloss_count':  int(merged_gloss_count),
    'merged_reduction':    int(merged_gloss_reduction),
    'safe_merge_count':    int(len(safe_merge_map)),
    'best_dev_wer':        float(best_wer),
    'test_wer_ensemble':   float(test_wer_ensemble) if len(ensemble_paths) >= 2 else None,
    'vocab_size':          int(num_classes),
    'vocab_after_merge':   int(num_classes),
    'history':             history,
}
with open(os.path.join(RESULTS_DIR, 'training_results.json'), 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print('\n✅ Risultati salvati.')

In [ ]:
# ============================================================
# BIGRAM LM RESCORING — nessun retraining necessario
# ============================================================
from collections import defaultdict

# Costruisci bigram dal train set
bigram_counts = defaultdict(lambda: defaultdict(int))
for seq in train_df['orth'].fillna(''):
    tokens = apply_merge(seq)
    ids = [g2i.get(t) for t in tokens if g2i.get(t) is not None]
    for a, b in zip(ids, ids[1:]):
        bigram_counts[a][b] += 1

log_bigram = {}
for prev, nexts in bigram_counts.items():
    total = sum(nexts.values())
    log_bigram[prev] = {nxt: np.log(cnt / total) for nxt, cnt in nexts.items()}

def greedy_decode_with_bigram(log_probs_TBC, beta=0.0, log_prior=None,
                               log_bigram=None, alpha=0.3):
    """Greedy decode con bigram rescoring: ri-ordina i top candidati per frame."""
    T, B, C = log_probs_TBC.shape
    results = []
    for b in range(B):
        lp = log_probs_TBC[:, b, :].clone()  # (T, C)
        if beta > 0.0 and log_prior is not None:
            lp = lp - beta * log_prior.unsqueeze(0)

        seq = []
        prev = None
        for t in range(T):
            if log_bigram and alpha > 0 and prev is not None and prev in log_bigram:
                # Combina score acustico + bigram
                bigram_scores = torch.tensor([
                    log_bigram[prev].get(c, -10.0) for c in range(C)
                ], dtype=torch.float32)
                combined = lp[t] + alpha * bigram_scores
                best = combined.argmax().item()
            else:
                best = lp[t].argmax().item()
            seq.append(best)
            if best != 0:
                prev = best

        results.append(collapse_ctc(seq))
    return results

# Cerca alpha ottimale sul dev set
print('Ricerca alpha bigram su dev set...')
best_alpha, best_alpha_wer = 0.0, float('inf')
model.eval()
for alpha in [0.0, 0.1, 0.2, 0.3, 0.5]:
    all_refs, all_hyps = [], []
    with torch.no_grad():
        for tssies, targets, input_lengths, target_lengths in dev_dl:
            tssies = tssies.to(DEVICE)
            lp = predict_with_tta(model, tssies).permute(1, 0, 2).float().cpu()
            decoded = greedy_decode_with_bigram(lp, beta=CONFIG['prior_beta'],
                                               log_prior=LOG_PRIOR.cpu(),
                                               log_bigram=log_bigram,
                                               alpha=alpha)
            all_hyps.extend(decoded)
            refs, offset = [], 0
            for tlen in target_lengths.tolist():
                all_refs.append(targets[offset:offset+tlen].tolist())
                offset += tlen
    wer = compute_wer(all_refs, all_hyps)
    print(f'  alpha={alpha:.1f} → Dev WER {wer*100:.2f}%')
    if wer < best_alpha_wer:
        best_alpha_wer, best_alpha = wer, alpha

print(f'✓ Best alpha: {best_alpha} (WER {best_alpha_wer*100:.2f}%)')
CONFIG['bigram_alpha'] = best_alpha  # salva separato, non sovrascrive prior_beta

## Step 12 — Visualizzazione

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Loss ─────────────────────────────────────────────────────────────────
epochs_all = range(1, len(history['train_loss']) + 1)
# colora le due fasi
phase1_end = CONFIG['phase1_epochs']
axes[0].axvspan(1, phase1_end, alpha=0.08, color='blue', label='Fase 1 (frozen)')
axes[0].axvspan(phase1_end, len(history['train_loss']), alpha=0.08, color='green', label='Fase 2 (full)')
axes[0].plot(epochs_all, history['train_loss'], label='Train Loss', alpha=0.8)
axes[0].plot(epochs_all, history['dev_loss'],   label='Dev Loss',   alpha=0.8)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('CTC Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ── WER ──────────────────────────────────────────────────────────────────
axes[1].axvspan(1, phase1_end, alpha=0.08, color='blue')
axes[1].axvspan(phase1_end, len(history['dev_wer']), alpha=0.08, color='green')
axes[1].plot(epochs_all, [w * 100 for w in history['dev_wer']],
             label='Dev WER (%)', color='red', alpha=0.8)
axes[1].axhline(best_wer * 100, color='green', linestyle='--',
                label=f'Best single: {best_wer*100:.1f}%')
if len(ensemble_paths) >= 2:
    axes[1].axhline(test_wer_ensemble * 100, color='purple', linestyle=':',
                    label=f'Ensemble test: {test_wer_ensemble*100:.1f}%')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('WER (%)')
axes[1].set_title('Word Error Rate (Validation)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n' + '=' * 60)
print('RIEPILOGO FINALE')
print('=' * 60)
print(f'Vocabolario raw:        {raw_gloss_count} gloss')
print(f'Vocabolario dopo merge: {merged_gloss_count} gloss')
print(f'Classi rimosse:         {merged_gloss_reduction}')
print(f'Merge sicuri applicati: {len(safe_merge_map)}')
print(f'LSTM layers:            {CONFIG["num_layers"]} (vs 2 originale)')
print(f'Prior β:                {CONFIG["prior_beta"]}')
print(f'Best dev WER:           {best_wer*100:.2f}%')
print(f'Test WER (singolo):     {test_wer_single*100:.2f}%')
if len(ensemble_paths) >= 2:
    print(f'Test WER (ensemble):    {test_wer_ensemble*100:.2f}%')
print('=' * 60)

In [ ]:
## TEST — Verifica Funzioni Ottimizzate

print("=" * 60)
print("TEST DELLE FUNZIONI OTTIMIZZATE")
print("=" * 60)

# Test 1: DFS order
print("\n✓ Test 1: DFS order corretto")
try:
    dfs = get_dfs_order_phoenix_correct()
    assert len(dfs) == 135, f"Expected 135 elements, got {len(dfs)}"
    assert dfs.dtype == np.int32, f"Expected int32, got {dfs.dtype}"
    print(f"  ✓ DFS order shape: {dfs.shape}, dtype: {dfs.dtype}")
except Exception as e:
    print(f"  ✗ ERRORE: {e}")

# Test 2: Augmentation
print("\n✓ Test 2: Augmentazione TSSI ottimizzata")
try:
    tssi = np.random.rand(5, 48, 100).astype(np.float32)
    tssi_aug = augment_tssi_fixed(tssi, augment=True)
    assert tssi_aug.shape == tssi.shape, f"Shape mismatch: {tssi_aug.shape}"
    assert tssi_aug.min() >= 0 and tssi_aug.max() <= 1.01, "TSSI not normalized"
    print(f"  ✓ Augmentation shape: {tssi_aug.shape}, min: {tssi_aug.min():.3f}, max: {tssi_aug.max():.3f}")
except Exception as e:
    print(f"  ✗ ERRORE: {e}")

# Test 3: CTC Loss with Entropy
print("\n✓ Test 3: CTC Loss with Entropy Regularization")
try:
    T, B, C = 50, 2, 10
    log_probs = torch.randn(T, B, C)
    targets = torch.tensor([1, 2, 3, 4, 1, 2], dtype=torch.long)
    
    criterion = CTCLossWithEntropy(blank=0, entropy_weight=0.05)
    loss = criterion(log_probs, targets,
                     input_lengths=torch.tensor([50, 50]),
                     target_lengths=torch.tensor([3, 3]))
    
    assert not torch.isnan(loss), "Loss is NaN!"
    print(f"  ✓ CTC Loss computed: {loss.item():.4f}")
except Exception as e:
    print(f"  ✗ ERRORE: {e}")

# Test 4: Beam Search Decoding
print("\n✓ Test 4: Beam Search CTC Decoding with Prior Scaling")
try:
    T, B, C = 20, 1, 5
    log_probs = torch.randn(T, B, C)
    log_prior = torch.randn(C)
    
    sequences = beam_search_ctc_optimized(log_probs, beam_width=3, beta=0.3, log_prior=log_prior)
    
    assert len(sequences) == B, f"Expected {B} sequences, got {len(sequences)}"
    assert all(1 <= c < C for seq in sequences for c in seq), "Invalid class indices"
    print(f"  ✓ Beam search decoded sequence (first 5): {sequences[0][:5] if sequences[0] else 'empty'}")
except Exception as e:
    print(f"  ✗ ERRORE: {e}")

# Test 5: Scheduler with Warmup
print("\n✓ Test 5: Scheduler con Warmup + Cosine Annealing")
try:
    model_test = torch.nn.Linear(10, 10)
    optimizer = torch.optim.Adam(model_test.parameters(), lr=1e-3)
    scheduler = create_scheduler_warmup_cosine(
        optimizer, 
        total_steps=1000,
        warmup_steps=100,
        min_lr_ratio=0.01
    )
    
    lrs = []
    for _ in range(150):
        lrs.append(optimizer.param_groups[0]['lr'])
        scheduler.step()
    
    assert lrs[0] < lrs[50], "Learning rate should increase during warmup"
    assert lrs[100] > lrs[110], "Learning rate should decrease after warmup"
    print(f"  ✓ Scheduler: LR warmup: {lrs[0]:.6f} → {lrs[50]:.6f}, then cosine: {lrs[100]:.6f} → {lrs[120]:.6f}")
except Exception as e:
    print(f"  ✗ ERRORE: {e}")

# Test 6: Checkpoint Manager
print("\n✓ Test 6: Checkpoint Manager")
try:
    import tempfile
    with tempfile.TemporaryDirectory() as tmpdir:
        ckpt_mgr = CheckpointManager(tmpdir, keep_top_k=2, metric='wer')
        
        model_test = torch.nn.Linear(10, 10)
        for epoch in range(3):
            path = ckpt_mgr.save(model_test, epoch, {'wer': 0.5 - epoch*0.1})
            assert path.exists(), f"Checkpoint not saved: {path}"
        
        assert len(ckpt_mgr.checkpoints) == 2, f"Should keep 2 checkpoints, got {len(ckpt_mgr.checkpoints)}"
        print(f"  ✓ CheckpointManager: Saved {len(ckpt_mgr.checkpoints)} checkpoint (top-2)")
except Exception as e:
    print(f"  ✗ ERRORE: {e}")

print("\n" + "=" * 60)
print("✅ TUTTI I TEST SONO PASSATI!")
print("=" * 60)